# Proyecto Integrador - Ciencia de Datos Ambientales

## Predicción de temperatura superficial del mar e identificación de olas de calor marinas en la costa del Perú (1990-2025)

Este notebook integra el flujo final del proyecto:

1. Definición del problema ambiental e hipotesis.
2. Ingesta y preparacion de datos NOAA OISST, con integracion opcional de ICEN y ERA5.
3. Analisis exploratorio de la temperatura superficial del mar.
4. Identificacion de olas de calor marinas mediante la metodologia de Hobday et al. (2016) usando SST observada.
5. Division regional en zona Norte, Centro y Sur.
6. Construccion de variables predictoras para estimar SST, incluyendo ICEN rezagado un mes si esta disponible.
7. Entrenamiento 70/30 de Random Forest Regressor con tres configuraciones por zona.
8. Comparacion de desempeno del modelo usando MAE, RMSE y R2.
9. Aplicacion de Hobday sobre la SST predicha para detectar MHW en el periodo de prueba.
10. Comparacion entre MHW observadas y MHW detectadas a partir de SST predicha.

**Nota metodologica:** como se trabaja con series temporales, la division 70/30 se hace en orden cronologico. El primer 70% de los dias se usa para entrenamiento y el 30% final se reserva para prueba. Esto evita mezclar informacion futura dentro del entrenamiento.

## 1. Descripcion del problema ambiental

Las olas de calor marinas (Marine Heatwaves, MHW) son periodos prolongados de temperatura superficial del mar anormalmente alta. En la costa peruana, estos eventos pueden alterar los ecosistemas marinos, afectar procesos de afloramiento, cambiar la disponibilidad de recursos pesqueros y modificar condiciones oceanograficas asociadas a eventos El Nino.

El proyecto busca predecir la temperatura superficial del mar (SST) frente a la costa peruana y, a partir de la SST predicha, aplicar la metodologia de Hobday et al. (2016) para identificar posibles olas de calor marinas.

### Indicadores ambientales clave

| Indicador | Unidad | Uso en el proyecto |
|---|---:|---|
| SST promedio regional/zonal | C | Variable oceanografica principal y variable objetivo del modelo |
| Anomalia climatologica de SST | C | Desviacion frente al comportamiento normal del dia del ano |
| Dias bajo condicion MHW | dias | Frecuencia anual o zonal de MHW |
| Duracion del evento | dias | Persistencia de cada ola de calor marina |
| Intensidad acumulada | C dia | Severidad total del evento |

### Hipotesis de trabajo

1. La SST puede ser estimada a partir de su comportamiento previo, variables estacionales y, si estan disponibles, variables oceanograficas/climaticas complementarias.
2. El desempeno del modelo Random Forest Regressor puede variar entre zona Norte, Centro y Sur debido a diferencias oceanograficas regionales.
3. Al aplicar Hobday sobre la SST predicha, es posible aproximar la ocurrencia de MHW en el periodo de prueba, aunque la deteccion dependera de la precision de la prediccion termica.

## 2. Instalacion y carga de librerias

Ejecutar esta celda en Google Colab. Si ya tienes instaladas las librerias, la instalacion sera rapida.

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 3. Arquitectura y flujo de datos

### Fuentes de datos

| Fuente | Formato | Variable principal | Uso |
|---|---|---|---|
| NOAA OISST v2.1 | NetCDF (.nc) | SST diaria | Base para deteccion MHW y variable objetivo del modelo |
| ICEN | CSV | Indice Costero El Nino | Variable explicativa opcional |
| ERA5 | NetCDF (.nc) | Viento u10/v10 | Variable explicativa opcional |

### Pipeline

Carga de datos -> limpieza y validacion -> promedio espacial por zona -> calculo de climatologia y umbral P90 -> deteccion MHW observada -> features predictoras -> prediccion de SST con Random Forest -> aplicacion de Hobday sobre SST predicha -> comparacion de detecciones -> conclusiones.

## 4. Conexion a Google Drive y rutas

Ajusta `BASE` si tus archivos estan en otra carpeta. El notebook funciona principalmente con NOAA OISST. ICEN y ERA5 se integran solo si existen en las rutas indicadas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
BASE = Path('/content/drive/MyDrive/DATOS_PROYECTO')

RUTA_NOAA = BASE / 'NOAA_OISST_PERU_RECORTADO_1990_2025'
RUTA_ICEN = BASE / 'ICEN' / 'ICEN_1990_2025.csv'
RUTA_ERA5_U = BASE / 'ERA5' / 'data_viento_u.nc'
RUTA_ERA5_V = BASE / 'ERA5' / 'data_viento_v.nc'

OUT_DIR = BASE / 'RESULTADOS_INTEGRADOR_FINAL'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Rutas configuradas:')
print('NOAA :', RUTA_NOAA)
print('ICEN :', RUTA_ICEN, '| existe:', RUTA_ICEN.exists())
print('ERA5 U:', RUTA_ERA5_U, '| existe:', RUTA_ERA5_U.exists())
print('ERA5 V:', RUTA_ERA5_V, '| existe:', RUTA_ERA5_V.exists())
print('Salida:', OUT_DIR)

## 5. Carga y preparacion de NOAA OISST

Se cargan los archivos mensuales `.nc`, se calcula la SST promedio para tres zonas latitudinales y tambien para el total del dominio peruano.

Zonas propuestas:

| Zona | Rango latitudinal aproximado |
|---|---|
| Norte | 0 a 6 S |
| Centro | 6 S a 13 S |
| Sur | 13 S a 20 S |

In [ ]:
archivos_noaa = sorted(glob.glob(str(RUTA_NOAA / 'oisst_peru_sst_*.nc')))

print('Total de archivos NOAA encontrados:', len(archivos_noaa))
if archivos_noaa:
    print('Primer archivo:', os.path.basename(archivos_noaa[0]))
    print('Ultimo archivo:', os.path.basename(archivos_noaa[-1]))

if len(archivos_noaa) == 0:
    raise FileNotFoundError('No se encontraron archivos NOAA. Revisa RUTA_NOAA.')
if len(archivos_noaa) != 432:
    print('Aviso: para 1990-2025 completos se esperaban 432 archivos mensuales.')

In [ ]:
def detectar_variable_sst(ds):
    nombres_preferidos = ['sst', 'SST', 'sea_surface_temperature', 'analysed_sst', 'temperature']
    for nombre in nombres_preferidos:
        if nombre in ds.data_vars:
            return nombre
    for nombre, da in ds.data_vars.items():
        if np.issubdtype(da.dtype, np.number) and any('time' in d.lower() for d in da.dims):
            return nombre
    for nombre, da in ds.data_vars.items():
        if np.issubdtype(da.dtype, np.number):
            return nombre
    raise ValueError('No se pudo detectar una variable numerica de SST.')


def detectar_dim_tiempo(da):
    for d in da.dims:
        if d.lower() in ['time', 't', 'valid_time'] or 'time' in d.lower():
            return d
    raise ValueError(f'No se pudo detectar la dimension temporal en dims={da.dims}')


def convertir_a_celsius(da):
    units = str(da.attrs.get('units', '')).lower()
    if 'kelvin' in units or units == 'k':
        return da - 273.15
    try:
        if float(da.mean(skipna=True).values) > 100:
            return da - 273.15
    except Exception:
        pass
    return da


def detectar_dim_lat_lon(da):
    lat_candidates = [d for d in da.dims if d.lower() in ['lat', 'latitude']]
    lon_candidates = [d for d in da.dims if d.lower() in ['lon', 'longitude']]
    if not lat_candidates or not lon_candidates:
        raise ValueError(f'No se detectaron dimensiones lat/lon en dims={da.dims}')
    return lat_candidates[0], lon_candidates[0]


ZONAS = {
    'Peru_total': None,
    'Norte':  {'lat_min': -6,  'lat_max': 0},
    'Centro': {'lat_min': -13, 'lat_max': -6},
    'Sur':    {'lat_min': -20, 'lat_max': -13},
}


def seleccionar_por_lat(da, lat_dim, lat_min, lat_max):
    lat_vals = da[lat_dim].values
    if lat_vals[0] <= lat_vals[-1]:
        return da.sel({lat_dim: slice(lat_min, lat_max)})
    return da.sel({lat_dim: slice(lat_max, lat_min)})


def leer_promedios_zonales_nc(ruta):
    ds = xr.open_dataset(ruta)
    var_sst = detectar_variable_sst(ds)
    da = convertir_a_celsius(ds[var_sst])
    dim_t = detectar_dim_tiempo(da)
    dim_lat, dim_lon = detectar_dim_lat_lon(da)

    resultados = []
    for zona, lim in ZONAS.items():
        if lim is None:
            sub = da
        else:
            sub = seleccionar_por_lat(da, dim_lat, lim['lat_min'], lim['lat_max'])

        dims_espaciales = [d for d in sub.dims if d != dim_t]
        serie = sub.mean(dim=dims_espaciales, skipna=True)
        df = serie.to_dataframe(name='sst_promedio').reset_index()
        if dim_t != 'time':
            df = df.rename(columns={dim_t: 'time'})
        df['fecha'] = pd.to_datetime(df['time']).dt.floor('D')
        df['zona'] = zona
        resultados.append(df[['fecha', 'zona', 'sst_promedio']])

    ds.close()
    return pd.concat(resultados, ignore_index=True)

In [ ]:
series = []
for f in tqdm(archivos_noaa, desc='Leyendo NOAA por zonas'):
    series.append(leer_promedios_zonales_nc(f))

df_sst_zonas = (
    pd.concat(series, ignore_index=True)
    .groupby(['fecha', 'zona'], as_index=False)['sst_promedio'].mean()
    .sort_values(['zona', 'fecha'])
    .reset_index(drop=True)
)

df_sst_zonas['anio'] = df_sst_zonas['fecha'].dt.year

display(df_sst_zonas.head())
print('Rango:', df_sst_zonas['fecha'].min().date(), 'a', df_sst_zonas['fecha'].max().date())
print('Zonas:', df_sst_zonas['zona'].unique())
print('Registros por zona:')
print(df_sst_zonas['zona'].value_counts().sort_index())

In [ ]:
# Validacion de continuidad temporal por zona
for zona, sub in df_sst_zonas.groupby('zona'):
    fechas_esperadas = pd.date_range(sub['fecha'].min(), sub['fecha'].max(), freq='D')
    faltantes = fechas_esperadas.difference(sub['fecha'])
    print(f'{zona:10s} | dias={len(sub):5d} | faltantes={len(faltantes)}')

## 6. Carga opcional de ICEN y ERA5

Estas fuentes no son obligatorias para que el modelo funcione. Si los archivos no existen, el notebook continua solo con variables derivadas de SST.

In [ ]:
def cargar_icen(ruta):
    if not Path(ruta).exists():
        print('ICEN no encontrado. Se continuara sin ICEN.')
        return None

    icen = pd.read_csv(ruta)
    icen.columns = [str(c).strip() for c in icen.columns]

    # Intento flexible de deteccion de columnas
    cols_lower = {c.lower(): c for c in icen.columns}
    col_anio = cols_lower.get('ano') or cols_lower.get('anio') or cols_lower.get('año') or cols_lower.get('year')
    col_mes = cols_lower.get('mes') or cols_lower.get('month')
    col_icen = cols_lower.get('icen')

    if col_anio is None or col_mes is None or col_icen is None:
        # Caso comun cuando las tres primeras columnas son ano, mes, ICEN
        icen = icen.iloc[:, :3].copy()
        icen.columns = ['anio', 'mes', 'icen']
    else:
        icen = icen[[col_anio, col_mes, col_icen]].copy()
        icen.columns = ['anio', 'mes', 'icen']

    icen = icen.dropna(subset=['anio', 'mes', 'icen']).copy()
    icen['anio'] = icen['anio'].astype(int)
    icen['mes'] = icen['mes'].astype(int)
    icen['fecha_mes'] = pd.to_datetime(
        icen['anio'].astype(str) + '-' + icen['mes'].astype(str).str.zfill(2) + '-01'
    )
    icen = icen[(icen['anio'] >= 1990) & (icen['anio'] <= 2025)].copy()
    icen = icen.sort_values('fecha_mes').reset_index(drop=True)

    # Para prediccion se usa el ICEN del mes anterior.
    # Asi evitamos usar informacion del mismo mes que podria no estar disponible
    # al momento de estimar la SST diaria.
    icen['icen_lag1'] = icen['icen'].shift(1)

    print('ICEN cargado:', len(icen), 'registros')
    return icen[['fecha_mes', 'anio', 'mes', 'icen', 'icen_lag1']]


def cargar_viento_era5(ruta_u, ruta_v):
    if not Path(ruta_u).exists() or not Path(ruta_v).exists():
        print('ERA5 no encontrado. Se continuara sin viento.')
        return None

    ds_u = xr.open_dataset(str(ruta_u))
    ds_v = xr.open_dataset(str(ruta_v))
    var_u = list(ds_u.data_vars)[0]
    var_v = list(ds_v.data_vars)[0]

    da_u = ds_u[var_u]
    da_v = ds_v[var_v]
    dim_t = detectar_dim_tiempo(da_u)
    dims_u = [d for d in da_u.dims if d != dim_t]
    dims_v = [d for d in da_v.dims if d != dim_t]

    u_prom = da_u.mean(dim=dims_u, skipna=True)
    v_prom = da_v.mean(dim=dims_v, skipna=True)

    df_viento = pd.DataFrame({
        'fecha': pd.to_datetime(u_prom[dim_t].values),
        'u10': np.asarray(u_prom.values).flatten(),
        'v10': np.asarray(v_prom.values).flatten(),
    })
    df_viento['fecha'] = df_viento['fecha'].dt.floor('D')
    df_viento = df_viento.groupby('fecha', as_index=False)[['u10', 'v10']].mean()
    df_viento['velocidad_viento'] = np.sqrt(df_viento['u10']**2 + df_viento['v10']**2)

    ds_u.close()
    ds_v.close()
    print('ERA5 cargado:', len(df_viento), 'dias')
    return df_viento


icen = cargar_icen(RUTA_ICEN)
df_viento = cargar_viento_era5(RUTA_ERA5_U, RUTA_ERA5_V)

## 7. Analisis descriptivo (EDA)

Se revisan valores faltantes, estadisticos generales, outliers y patrones temporales por zona.

In [ ]:
print('Valores faltantes por columna:')
print(df_sst_zonas.isna().sum())

print('\nEstadisticas de SST por zona:')
display(df_sst_zonas.groupby('zona')['sst_promedio'].describe().round(3))

In [ ]:
# Deteccion de outliers por zona con criterio 3*IQR
outliers_resumen = []
for zona, sub in df_sst_zonas.groupby('zona'):
    q1 = sub['sst_promedio'].quantile(0.25)
    q3 = sub['sst_promedio'].quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 3 * iqr
    limite_sup = q3 + 3 * iqr
    n_out = ((sub['sst_promedio'] < limite_inf) | (sub['sst_promedio'] > limite_sup)).sum()
    outliers_resumen.append({
        'zona': zona, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'limite_inf': limite_inf, 'limite_sup': limite_sup, 'outliers': int(n_out)
    })

outliers_resumen = pd.DataFrame(outliers_resumen)
display(outliers_resumen.round(3))

In [ ]:
# Serie mensual por zona
sst_mensual_zonas = (
    df_sst_zonas
    .set_index('fecha')
    .groupby('zona')['sst_promedio']
    .resample('MS')
    .mean()
    .reset_index()
)

plt.figure(figsize=(15, 5))
for zona in ['Norte', 'Centro', 'Sur', 'Peru_total']:
    sub = sst_mensual_zonas[sst_mensual_zonas['zona'] == zona]
    plt.plot(sub['fecha'], sub['sst_promedio'], label=zona, linewidth=1)

plt.title('SST mensual promedio por zona - Costa del Peru')
plt.xlabel('Fecha')
plt.ylabel('SST (C)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Anomalias mensuales por zona
sst_mensual_zonas['mes'] = sst_mensual_zonas['fecha'].dt.month
clim_mes_zona = (
    sst_mensual_zonas
    .groupby(['zona', 'mes'])['sst_promedio']
    .mean()
    .rename('climatologia_mes')
    .reset_index()
)
sst_mensual_zonas = sst_mensual_zonas.merge(clim_mes_zona, on=['zona', 'mes'], how='left')
sst_mensual_zonas['anomalia_mensual'] = sst_mensual_zonas['sst_promedio'] - sst_mensual_zonas['climatologia_mes']

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for ax, zona in zip(axes, ['Norte', 'Centro', 'Sur']):
    sub = sst_mensual_zonas[sst_mensual_zonas['zona'] == zona]
    ax.axhline(0, color='black', linewidth=0.8)
    ax.fill_between(sub['fecha'], sub['anomalia_mensual'], 0,
                    where=sub['anomalia_mensual'] >= 0, color='tomato', alpha=0.7)
    ax.fill_between(sub['fecha'], sub['anomalia_mensual'], 0,
                    where=sub['anomalia_mensual'] < 0, color='steelblue', alpha=0.7)
    ax.set_title(f'Anomalia mensual de SST - Zona {zona}')
    ax.set_ylabel('Anomalia (C)')
axes[-1].set_xlabel('Fecha')
plt.tight_layout()
plt.show()

## 8. Deteccion de olas de calor marinas - metodologia Hobday et al. (2016)

Criterios aplicados:

1. Calculo de climatologia diaria sobre un periodo base.
2. Umbral diario del percentil 90.
3. Dia candidato MHW cuando la SST supera el umbral P90.
4. Evento MHW cuando la condicion dura al menos 5 dias.
5. Union de eventos separados por brechas cortas de hasta 2 dias.

In [ ]:
BASELINE = (1990, 2025)
PERCENTIL = 90
VENTANA_DIAS = 11
SUAVIZADO_DIAS = 31
DURACION_MINIMA = 5
MAX_GAP = 2

print('Parametros:')
print('Baseline:', BASELINE)
print('Percentil:', PERCENTIL)
print('Ventana:', VENTANA_DIAS)
print('Suavizado:', SUAVIZADO_DIAS)
print('Duracion minima:', DURACION_MINIMA)
print('Max gap:', MAX_GAP)

In [ ]:
def doy_sin_bisiesto(fechas):
    fechas = pd.to_datetime(fechas)
    doy = fechas.dt.dayofyear.astype(float)
    es_bisiesto = fechas.dt.is_leap_year
    despues_feb = fechas.dt.month > 2
    es_29feb = (fechas.dt.month == 2) & (fechas.dt.day == 29)
    doy.loc[es_bisiesto & despues_feb] -= 1
    doy.loc[es_29feb] = np.nan
    return doy


def suavizado_circular(valores, ventana=31):
    valores = np.asarray(valores, dtype=float)
    pad = ventana // 2
    extendido = np.r_[valores[-pad:], valores, valores[:pad]]
    suavizado = (
        pd.Series(extendido)
        .rolling(window=ventana, center=True, min_periods=1)
        .mean()
        .iloc[pad:pad + len(valores)]
        .to_numpy()
    )
    return suavizado


def calcular_climatologia_hobday_zona(df_zona, baseline=BASELINE, percentil=PERCENTIL,
                                      ventana_dias=VENTANA_DIAS, suavizado_dias=SUAVIZADO_DIAS):
    df = df_zona.copy().sort_values('fecha')
    df['doy'] = doy_sin_bisiesto(df['fecha'])
    df = df.dropna(subset=['doy']).copy()
    df['doy'] = df['doy'].astype(int)
    df['anio'] = df['fecha'].dt.year

    base = df[(df['anio'] >= baseline[0]) & (df['anio'] <= baseline[1])].copy()
    if base.empty:
        raise ValueError('El periodo baseline no tiene datos para esta zona.')

    mitad = ventana_dias // 2
    registros = []
    for d in range(1, 366):
        distancia = ((base['doy'] - d + 182) % 365) - 182
        muestra = base.loc[distancia.abs() <= mitad, 'sst_promedio'].dropna()
        registros.append({
            'doy': d,
            'climatologia': muestra.mean(),
            'umbral_p90': np.nanpercentile(muestra, percentil),
            'n_datos_base': len(muestra),
        })

    clim = pd.DataFrame(registros)
    clim['climatologia'] = suavizado_circular(clim['climatologia'], ventana=suavizado_dias)
    clim['umbral_p90'] = suavizado_circular(clim['umbral_p90'], ventana=suavizado_dias)
    return clim


def unir_brechas_cortas(condicion, max_gap=2):
    arr = np.asarray(condicion, dtype=bool).copy()
    n = len(arr)
    i = 0
    while i < n:
        if arr[i]:
            i += 1
            continue
        inicio = i
        while i < n and not arr[i]:
            i += 1
        fin = i - 1
        largo = fin - inicio + 1
        hay_antes = inicio > 0 and arr[inicio - 1]
        hay_despues = i < n and arr[i]
        if hay_antes and hay_despues and largo <= max_gap:
            arr[inicio:fin + 1] = True
    return arr


def detectar_eventos_mhw_zona(df_zona, zona, min_duracion=DURACION_MINIMA, max_gap=MAX_GAP):
    df = df_zona.sort_values('fecha').reset_index(drop=True).copy()
    condicion_unida = unir_brechas_cortas(df['sobre_umbral'].fillna(False).values, max_gap=max_gap)
    df['mhw_candidato'] = condicion_unida
    df['mhw'] = False
    df['evento_id_zona'] = np.nan

    eventos = []
    evento_id = 1
    n = len(df)
    i = 0

    while i < n:
        if not condicion_unida[i]:
            i += 1
            continue

        inicio = i
        while i < n and condicion_unida[i]:
            i += 1
        fin = i - 1
        duracion = fin - inicio + 1

        if duracion >= min_duracion:
            idx = np.arange(inicio, fin + 1)
            sub = df.iloc[idx].copy()
            idx_pico = sub['anomalia_climatologica'].idxmax()
            fila_pico = df.loc[idx_pico]

            df.loc[idx, 'mhw'] = True
            df.loc[idx, 'evento_id_zona'] = evento_id

            eventos.append({
                'zona': zona,
                'evento_id_zona': evento_id,
                'fecha_inicio': sub['fecha'].min(),
                'fecha_fin': sub['fecha'].max(),
                'anio_inicio': sub['fecha'].min().year,
                'duracion_dias': duracion,
                'dias_sobre_umbral': int(sub['sobre_umbral'].sum()),
                'fecha_pico': fila_pico['fecha'],
                'sst_pico': fila_pico['sst_promedio'],
                'intensidad_maxima': sub['anomalia_climatologica'].max(),
                'intensidad_media': sub['anomalia_climatologica'].mean(),
                'intensidad_acumulada': sub['anomalia_climatologica'].sum(),
                'exceso_umbral_maximo': sub['exceso_umbral'].max(),
                'exceso_umbral_medio': sub['exceso_umbral'].mean(),
            })
            evento_id += 1

    return df, pd.DataFrame(eventos)

In [ ]:
# Calcular climatologia y detectar eventos para cada zona
mhw_diario_lista = []
eventos_lista = []
clim_lista = []

for zona, sub in tqdm(df_sst_zonas.groupby('zona'), desc='Detectando MHW por zona'):
    clim_z = calcular_climatologia_hobday_zona(sub)
    clim_z['zona'] = zona
    clim_lista.append(clim_z)

    tmp = sub.copy()
    tmp['doy'] = doy_sin_bisiesto(tmp['fecha'])
    tmp = tmp.dropna(subset=['doy']).copy()
    tmp['doy'] = tmp['doy'].astype(int)
    tmp = tmp.merge(clim_z[['doy', 'climatologia', 'umbral_p90']], on='doy', how='left')
    tmp['anomalia_climatologica'] = tmp['sst_promedio'] - tmp['climatologia']
    tmp['exceso_umbral'] = tmp['sst_promedio'] - tmp['umbral_p90']
    tmp['sobre_umbral'] = tmp['sst_promedio'] > tmp['umbral_p90']

    mhw_z, eventos_z = detectar_eventos_mhw_zona(tmp, zona=zona)
    mhw_diario_lista.append(mhw_z)
    eventos_lista.append(eventos_z)

climatologia_zonas = pd.concat(clim_lista, ignore_index=True)
mhw_diario = pd.concat(mhw_diario_lista, ignore_index=True)
eventos_mhw = pd.concat(eventos_lista, ignore_index=True)

# ID global para exportacion
if len(eventos_mhw) > 0:
    eventos_mhw = eventos_mhw.sort_values(['zona', 'fecha_inicio']).reset_index(drop=True)
    eventos_mhw['evento_id_global'] = np.arange(1, len(eventos_mhw) + 1)

print('Eventos detectados por zona:')
print(eventos_mhw.groupby('zona').size().sort_index())
display(eventos_mhw.head())

In [ ]:
# Resumen anual por zona
resumen_anual = (
    mhw_diario
    .groupby(['zona', mhw_diario['fecha'].dt.year])
    .agg(
        sst_media=('sst_promedio', 'mean'),
        sst_max=('sst_promedio', 'max'),
        dias_mhw=('mhw', 'sum'),
        eventos_mhw=('evento_id_zona', lambda x: x.dropna().nunique()),
        intensidad_maxima_anual=('anomalia_climatologica', 'max'),
        intensidad_acumulada_mhw=('anomalia_climatologica', lambda x: x[mhw_diario.loc[x.index, 'mhw']].sum())
    )
    .reset_index()
    .rename(columns={'fecha': 'anio'})
)
resumen_anual['dias_mhw'] = resumen_anual['dias_mhw'].astype(int)
resumen_anual['eventos_mhw'] = resumen_anual['eventos_mhw'].astype(int)

display(resumen_anual.sort_values(['zona', 'dias_mhw'], ascending=[True, False]).groupby('zona').head(5))

In [ ]:
# Comparacion anual de dias MHW por zona
plt.figure(figsize=(15, 6))
for zona in ['Norte', 'Centro', 'Sur']:
    sub = resumen_anual[resumen_anual['zona'] == zona]
    plt.plot(sub['anio'], sub['dias_mhw'], marker='o', linewidth=1.5, label=zona)
plt.title('Días MHW por año y zona')
plt.xlabel('Año')
plt.ylabel('Días MHW')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Clasificacion de eventos MHW

Se clasifica cada evento segun su intensidad acumulada. Esta clasificacion resume la severidad total del evento porque integra la anomalia de SST durante todos los dias de duracion.

In [ ]:
def clasificar_intensidad_acumulada(valor):
    if valor < 25:
        return 'I - Baja (<25 C dia)'
    elif valor < 100:
        return 'II - Moderada (25-99 C dia)'
    elif valor < 200:
        return 'III - Alta (100-199 C dia)'
    return 'IV - Extrema (>=200 C dia)'


def clasificar_duracion(dias):
    if dias < 30:
        return 'Corta (5-29 dias)'
    elif dias < 100:
        return 'Media (30-99 dias)'
    return 'Larga (>=100 dias)'


eventos_mhw['categoria_intensidad'] = eventos_mhw['intensidad_acumulada'].apply(clasificar_intensidad_acumulada)
eventos_mhw['categoria_duracion'] = eventos_mhw['duracion_dias'].apply(clasificar_duracion)

display(eventos_mhw[[
    'zona', 'evento_id_zona', 'fecha_inicio', 'fecha_fin', 'duracion_dias',
    'intensidad_acumulada', 'categoria_intensidad', 'categoria_duracion'
]].head(20))

In [ ]:
# Distribucion de categorias por zona
cat_zona = pd.crosstab(eventos_mhw['zona'], eventos_mhw['categoria_intensidad'])
display(cat_zona)

cat_zona.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='Reds')
plt.title('Categorias de intensidad acumulada por zona')
plt.xlabel('Zona')
plt.ylabel('Numero de eventos')
plt.legend(title='Categoria', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Ingenieria de variables para predecir SST

El modelo no predice directamente si ocurre una ola de calor marina. El modelo predice la **temperatura superficial del mar (`sst_promedio`)**.

Luego, con la SST predicha, se aplica la metodologia de Hobday:

1. `anomalia_predicha = sst_predicha - climatologia`
2. `sobre_umbral_predicho = sst_predicha > umbral_p90`
3. Se identifican secuencias de al menos 5 dias sobre el umbral.
4. Se calculan eventos MHW predichos y se comparan con los eventos observados.

Para evitar fuga de informacion, las variables predictoras usan informacion previa: rezagos, medias moviles previas y variables temporales conocidas. Si ICEN esta disponible, se usa como `icen_lag1`, es decir, el ICEN del mes anterior.

In [ ]:
# Construir base diaria para ML
ml = mhw_diario.copy().sort_values(['zona', 'fecha']).reset_index(drop=True)
ml['anio'] = ml['fecha'].dt.year
ml['mes'] = ml['fecha'].dt.month
ml['doy_calendario'] = ml['fecha'].dt.dayofyear
ml['doy_sin'] = np.sin(2 * np.pi * ml['doy'] / 365)
ml['doy_cos'] = np.cos(2 * np.pi * ml['doy'] / 365)

# Integrar ICEN mensual rezagado si existe.
# Cada dia recibe el ICEN del mes anterior (icen_lag1), no el ICEN del mismo mes.
if icen is not None:
    ml['fecha_mes'] = ml['fecha'].dt.to_period('M').dt.to_timestamp()
    ml = ml.merge(icen[['fecha_mes', 'icen_lag1']], on='fecha_mes', how='left')
    ml = ml.drop(columns=['fecha_mes'])
else:
    ml['icen_lag1'] = np.nan

# Integrar viento diario si existe
if df_viento is not None:
    ml = ml.merge(df_viento, on='fecha', how='left')
else:
    ml['u10'] = np.nan
    ml['v10'] = np.nan
    ml['velocidad_viento'] = np.nan

# Variables rezagadas por zona: informacion anterior al dia que se predice
for lag in [1, 3, 7, 14, 30]:
    ml[f'sst_lag{lag}'] = ml.groupby('zona')['sst_promedio'].shift(lag)
    ml[f'anom_lag{lag}'] = ml.groupby('zona')['anomalia_climatologica'].shift(lag)
    ml[f'exceso_lag{lag}'] = ml.groupby('zona')['exceso_umbral'].shift(lag)

# Medias moviles previas. shift(1) evita incluir el dia objetivo.
ml['sst_ma7_prev'] = ml.groupby('zona')['sst_promedio'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
ml['sst_ma30_prev'] = ml.groupby('zona')['sst_promedio'].transform(lambda s: s.shift(1).rolling(30, min_periods=10).mean())
ml['anom_ma7_prev'] = ml.groupby('zona')['anomalia_climatologica'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
ml['exceso_ma7_prev'] = ml.groupby('zona')['exceso_umbral'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())

# Rezagos de viento si ERA5 esta disponible
for col in ['u10', 'v10', 'velocidad_viento']:
    if col in ml.columns:
        ml[f'{col}_lag1'] = ml.groupby('zona')[col].shift(1)
        ml[f'{col}_lag7'] = ml.groupby('zona')[col].shift(7)

FEATURES_CANDIDATAS = [
    'mes', 'doy_sin', 'doy_cos', 'climatologia', 'umbral_p90',
    'sst_lag1', 'sst_lag3', 'sst_lag7', 'sst_lag14', 'sst_lag30',
    'anom_lag1', 'anom_lag3', 'anom_lag7', 'anom_lag14', 'anom_lag30',
    'exceso_lag1', 'exceso_lag3', 'exceso_lag7', 'exceso_lag14', 'exceso_lag30',
    'sst_ma7_prev', 'sst_ma30_prev', 'anom_ma7_prev', 'exceso_ma7_prev',
    'icen_lag1', 'u10_lag1', 'u10_lag7', 'v10_lag1', 'v10_lag7',
    'velocidad_viento_lag1', 'velocidad_viento_lag7'
]

# Se conservan variables con suficiente disponibilidad de datos
feature_cols = []
for col in FEATURES_CANDIDATAS:
    if col in ml.columns and ml[col].notna().mean() >= 0.70:
        feature_cols.append(col)

TARGET_SST = 'sst_promedio'

columnas_base = [
    'fecha', 'zona', 'sst_promedio', 'climatologia', 'umbral_p90',
    'anomalia_climatologica', 'exceso_umbral', 'sobre_umbral', 'mhw'
]

# Evitar columnas duplicadas: algunas variables, como climatologia y umbral_p90,
# estan en columnas_base y tambien pueden usarse como features del modelo.
columnas_modelo = columnas_base + [c for c in feature_cols if c not in columnas_base]

df_ml = ml[columnas_modelo].dropna().copy()
df_ml['mhw'] = df_ml['mhw'].astype(int)

# Verificacion de seguridad: no deben quedar columnas duplicadas.
assert not df_ml.columns.duplicated().any(), 'Hay columnas duplicadas en df_ml'

print('Variable objetivo:', TARGET_SST)
print('Features usadas:', feature_cols)
print('Registros ML:', len(df_ml))
print('Registros por zona:')
print(df_ml['zona'].value_counts().sort_index())
display(df_ml.head())

In [ ]:
# Correlacion exploratoria de features con SST por zona
for zona in ['Norte', 'Centro', 'Sur']:
    sub = df_ml[df_ml['zona'] == zona]
    corr = sub[feature_cols + [TARGET_SST]].corr()[TARGET_SST].drop(TARGET_SST).sort_values()
    plt.figure(figsize=(9, 6))
    corr.plot(kind='barh', color=['steelblue' if v < 0 else 'tomato' for v in corr])
    plt.title(f'Correlacion de features con SST - Zona {zona}')
    plt.xlabel('Correlacion de Pearson')
    plt.tight_layout()
    plt.show()

## 11. Random Forest Regressor con tres configuraciones

Se entrenan tres configuraciones para predecir SST en cada zona. La division 70/30 es cronologica:

- **Entrenamiento:** primer 70% de registros.
- **Prueba:** ultimo 30% de registros.

Metricas principales del modelo de temperatura:

| Metrica | Interpretacion |
|---|---|
| MAE | Error absoluto promedio en grados C |
| RMSE | Error cuadratico medio, penaliza mas errores grandes |
| R2 | Proporcion de variabilidad de SST explicada por el modelo |

Despues se evalua la deteccion MHW derivada de la SST predicha usando precision, recall y F1 de la clase MHW.

In [ ]:
RF_CONFIGS = {
    'RF_1_conservador': {
        'n_estimators': 200,
        'max_depth': 8,
        'min_samples_leaf': 8,
        'min_samples_split': 10,
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1,
    },
    'RF_2_intermedio': {
        'n_estimators': 400,
        'max_depth': 14,
        'min_samples_leaf': 4,
        'min_samples_split': 8,
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1,
    },
    'RF_3_profundo': {
        'n_estimators': 600,
        'max_depth': None,
        'min_samples_leaf': 2,
        'min_samples_split': 4,
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1,
    },
}


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def detectar_mhw_desde_sst_predicha(df_test, sst_predicha, zona, modelo_nombre):
    """Aplica Hobday sobre SST predicha dentro del periodo de prueba."""
    pred = df_test[['fecha', 'zona', 'climatologia', 'umbral_p90']].copy()
    pred['sst_promedio'] = sst_predicha
    pred['anomalia_climatologica'] = pred['sst_promedio'] - pred['climatologia']
    pred['exceso_umbral'] = pred['sst_promedio'] - pred['umbral_p90']
    pred['sobre_umbral'] = pred['sst_promedio'] > pred['umbral_p90']

    diario_pred, eventos_pred = detectar_eventos_mhw_zona(
        pred, zona=f'{zona}_{modelo_nombre}',
        min_duracion=DURACION_MINIMA,
        max_gap=MAX_GAP
    )
    diario_pred = diario_pred.rename(columns={
        'sst_promedio': 'sst_predicha',
        'anomalia_climatologica': 'anomalia_predicha',
        'exceso_umbral': 'exceso_umbral_predicho',
        'sobre_umbral': 'sobre_umbral_predicho',
        'mhw': 'mhw_predicho'
    })
    diario_pred['zona_original'] = zona
    diario_pred['modelo'] = modelo_nombre

    if len(eventos_pred) > 0:
        eventos_pred['zona_original'] = zona
        eventos_pred['modelo'] = modelo_nombre

    return diario_pred, eventos_pred


def evaluar_regresores_por_zona(df, zona, feature_cols, target='sst_promedio', train_size=0.70):
    data = df[df['zona'] == zona].sort_values('fecha').reset_index(drop=True).copy()
    split_idx = int(len(data) * train_size)

    train = data.iloc[:split_idx].copy()
    test = data.iloc[split_idx:].copy()

    X_train = train[feature_cols]
    y_train = train[target]
    X_test = test[feature_cols]
    y_test = test[target]

    resultados = []
    modelos = {}
    predicciones = test[[
        'fecha', 'zona', 'sst_promedio', 'climatologia', 'umbral_p90',
        'anomalia_climatologica', 'exceso_umbral', 'mhw'
    ]].copy().rename(columns={'sst_promedio': 'sst_observada', 'mhw': 'mhw_observada'})

    diarios_predichos = []
    eventos_predichos = []

    for nombre, params in RF_CONFIGS.items():
        modelo = RandomForestRegressor(**params)
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        diario_pred, eventos_pred = detectar_mhw_desde_sst_predicha(test, y_pred, zona, nombre)

        # Unir mhw_predicho por fecha para comparacion binaria
        comp = predicciones[['fecha', 'mhw_observada']].merge(
            diario_pred[['fecha', 'mhw_predicho', 'sst_predicha']], on='fecha', how='left'
        )
        comp['mhw_predicho'] = comp['mhw_predicho'].fillna(False).astype(int)
        comp['mhw_observada'] = comp['mhw_observada'].astype(int)

        resultados.append({
            'zona': zona,
            'modelo': nombre,
            'n_train': len(train),
            'n_test': len(test),
            'inicio_train': train['fecha'].min(),
            'fin_train': train['fecha'].max(),
            'inicio_test': test['fecha'].min(),
            'fin_test': test['fecha'].max(),
            'mae_sst': mean_absolute_error(y_test, y_pred),
            'rmse_sst': rmse_score(y_test, y_pred),
            'r2_sst': r2_score(y_test, y_pred),
            'bias_sst': np.mean(y_pred - y_test),
            'dias_mhw_observados_test': int(comp['mhw_observada'].sum()),
            'dias_mhw_predichos_test': int(comp['mhw_predicho'].sum()),
            'eventos_mhw_predichos_test': int(len(eventos_pred)),
            'precision_mhw_derivada': precision_score(comp['mhw_observada'], comp['mhw_predicho'], zero_division=0),
            'recall_mhw_derivada': recall_score(comp['mhw_observada'], comp['mhw_predicho'], zero_division=0),
            'f1_mhw_derivada': f1_score(comp['mhw_observada'], comp['mhw_predicho'], zero_division=0),
            'accuracy_mhw_derivada': accuracy_score(comp['mhw_observada'], comp['mhw_predicho']),
        })

        modelos[nombre] = modelo
        predicciones[f'sst_pred_{nombre}'] = y_pred
        predicciones[f'mhw_pred_{nombre}'] = comp['mhw_predicho'].values
        diarios_predichos.append(diario_pred)
        eventos_predichos.append(eventos_pred)

    diarios_predichos = pd.concat(diarios_predichos, ignore_index=True)
    eventos_predichos = pd.concat([e for e in eventos_predichos if len(e) > 0], ignore_index=True) if any(len(e) > 0 for e in eventos_predichos) else pd.DataFrame()

    return pd.DataFrame(resultados), modelos, predicciones, diarios_predichos, eventos_predichos, (X_train, X_test, y_train, y_test)

In [ ]:
zonas_modelo = ['Norte', 'Centro', 'Sur']
resultados_lista = []
modelos_por_zona = {}
predicciones_lista = []
diarios_predichos_lista = []
eventos_predichos_lista = []
splits_por_zona = {}

for zona in zonas_modelo:
    res_z, modelos_z, pred_z, diarios_z, eventos_z, split_z = evaluar_regresores_por_zona(
        df_ml, zona, feature_cols, target=TARGET_SST, train_size=0.70
    )
    resultados_lista.append(res_z)
    modelos_por_zona[zona] = modelos_z
    predicciones_lista.append(pred_z)
    diarios_predichos_lista.append(diarios_z)
    if len(eventos_z) > 0:
        eventos_predichos_lista.append(eventos_z)
    splits_por_zona[zona] = split_z

tabla_modelos = pd.concat(resultados_lista, ignore_index=True)
predicciones_test = pd.concat(predicciones_lista, ignore_index=True)
diario_predicho_modelos = pd.concat(diarios_predichos_lista, ignore_index=True)
eventos_predichos_modelos = pd.concat(eventos_predichos_lista, ignore_index=True) if eventos_predichos_lista else pd.DataFrame()

cols_metricas = [
    'zona', 'modelo', 'n_train', 'n_test', 'inicio_train', 'fin_train', 'inicio_test', 'fin_test',
    'mae_sst', 'rmse_sst', 'r2_sst', 'bias_sst',
    'dias_mhw_observados_test', 'dias_mhw_predichos_test', 'eventos_mhw_predichos_test',
    'precision_mhw_derivada', 'recall_mhw_derivada', 'f1_mhw_derivada', 'accuracy_mhw_derivada'
]

display(tabla_modelos[cols_metricas].round(4))

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE DESEMPEÑO: MAE, RMSE Y R2
# Por zona y por configuración de Random Forest
# ============================================================

tabla_desempeno_sst = tabla_modelos[
    ['zona', 'modelo', 'mae_sst', 'rmse_sst', 'r2_sst']
].copy()

# Redondear para presentación
tabla_desempeno_sst['mae_sst'] = tabla_desempeno_sst['mae_sst'].round(4)
tabla_desempeno_sst['rmse_sst'] = tabla_desempeno_sst['rmse_sst'].round(4)
tabla_desempeno_sst['r2_sst'] = tabla_desempeno_sst['r2_sst'].round(4)

# Ordenar por zona y por menor RMSE
tabla_desempeno_sst = tabla_desempeno_sst.sort_values(
    by=['zona', 'rmse_sst'],
    ascending=[True, True]
).reset_index(drop=True)

display(tabla_desempeno_sst)

In [ ]:
# Seleccion del mejor modelo por zona segun RMSE de SST
mejores_por_zona = (
    tabla_modelos
    .sort_values(['zona', 'rmse_sst', 'mae_sst', 'r2_sst'], ascending=[True, True, True, False])
    .groupby('zona')
    .head(1)
    .reset_index(drop=True)
)

print('Mejor configuracion por zona segun RMSE de SST:')
display(mejores_por_zona[[
    'zona', 'modelo', 'mae_sst', 'rmse_sst', 'r2_sst',
    'precision_mhw_derivada', 'recall_mhw_derivada', 'f1_mhw_derivada'
]].round(4))

mejor_global = tabla_modelos.sort_values(['rmse_sst', 'mae_sst'], ascending=True).iloc[0]
print('\nMejor resultado global segun RMSE:')
print(mejor_global[['zona', 'modelo', 'mae_sst', 'rmse_sst', 'r2_sst', 'f1_mhw_derivada']])

In [ ]:
# Gráficos comparativos de las tres configuraciones
# Solo métricas principales de regresión SST

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE
sns.barplot(
    data=tabla_modelos,
    x='zona',
    y='rmse_sst',
    hue='modelo',
    ax=axes[0]
)
axes[0].set_title('RMSE de SST por zona y configuración')
axes[0].set_ylabel('RMSE SST (°C)')
axes[0].set_xlabel('Zona')

# R2
sns.barplot(
    data=tabla_modelos,
    x='zona',
    y='r2_sst',
    hue='modelo',
    ax=axes[1]
)
axes[1].set_title('R² de SST por zona y configuración')
axes[1].set_ylabel('R² SST')
axes[1].set_xlabel('Zona')
axes[1].legend_.remove()

# MAE
sns.barplot(
    data=tabla_modelos,
    x='zona',
    y='mae_sst',
    hue='modelo',
    ax=axes[2]
)
axes[2].set_title('MAE de SST por zona y configuración')
axes[2].set_ylabel('MAE SST (°C)')
axes[2].set_xlabel('Zona')
axes[2].legend_.remove()

axes[0].legend(title='Modelo', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.suptitle(
    'Comparación de configuraciones Random Forest para predicción de SST',
    fontsize=14,
    y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables para el mejor modelo de cada zona
importancias = []
for _, row in mejores_por_zona.iterrows():
    zona = row['zona']
    modelo_nombre = row['modelo']
    modelo = modelos_por_zona[zona][modelo_nombre]
    imp = pd.DataFrame({
        'zona': zona,
        'modelo': modelo_nombre,
        'variable': feature_cols,
        'importancia': modelo.feature_importances_,
    }).sort_values('importancia', ascending=False)
    importancias.append(imp)

importancias_mejores = pd.concat(importancias, ignore_index=True)

g = sns.catplot(
    data=importancias_mejores.groupby('zona').head(12),
    x='importancia', y='variable', col='zona', kind='bar',
    sharey=False, height=5, aspect=0.9, color='steelblue'
)
g.fig.suptitle('Top variables para predecir SST - mejor Random Forest por zona', y=1.03)
plt.show()

display(importancias_mejores.groupby('zona').head(10))

## 12. Aplicacion de Hobday sobre la SST predicha

En esta seccion se toma el mejor modelo de SST por zona y se conserva una sola serie predicha por zona. Sobre esa serie se aplica la misma logica de deteccion MHW usada para los datos observados.

Esto permite responder la pregunta del proyecto de manera metodologicamente correcta:

**Primero se predice SST. Luego se detectan olas de calor marinas a partir de la SST predicha.**

In [ ]:
predicciones_mejor_modelo = []
eventos_predichos_mejor = []
diario_predicho_mejor = []

for _, row in mejores_por_zona.iterrows():
    zona = row['zona']
    modelo_nombre = row['modelo']

    pred_z = predicciones_test[predicciones_test['zona'] == zona].copy()
    pred_z['modelo_seleccionado'] = modelo_nombre
    pred_z['sst_predicha'] = pred_z[f'sst_pred_{modelo_nombre}']
    pred_z['mhw_predicha'] = pred_z[f'mhw_pred_{modelo_nombre}']
    pred_z['error_sst'] = pred_z['sst_predicha'] - pred_z['sst_observada']
    pred_z['anomalia_predicha'] = pred_z['sst_predicha'] - pred_z['climatologia']
    pred_z['exceso_umbral_predicho'] = pred_z['sst_predicha'] - pred_z['umbral_p90']
    predicciones_mejor_modelo.append(pred_z[[
        'fecha', 'zona', 'modelo_seleccionado', 'sst_observada', 'sst_predicha', 'error_sst',
        'climatologia', 'umbral_p90', 'mhw_observada', 'mhw_predicha',
        'anomalia_climatologica', 'anomalia_predicha', 'exceso_umbral', 'exceso_umbral_predicho'
    ]])

    diario_z = diario_predicho_modelos[
        (diario_predicho_modelos['zona_original'] == zona) &
        (diario_predicho_modelos['modelo'] == modelo_nombre)
    ].copy()
    diario_predicho_mejor.append(diario_z)

    if len(eventos_predichos_modelos) > 0:
        eventos_z = eventos_predichos_modelos[
            (eventos_predichos_modelos['zona_original'] == zona) &
            (eventos_predichos_modelos['modelo'] == modelo_nombre)
        ].copy()
        if len(eventos_z) > 0:
            eventos_predichos_mejor.append(eventos_z)

predicciones_mejor_modelo = pd.concat(predicciones_mejor_modelo, ignore_index=True)
diario_predicho_mejor = pd.concat(diario_predicho_mejor, ignore_index=True)
eventos_predichos_mejor = pd.concat(eventos_predichos_mejor, ignore_index=True) if eventos_predichos_mejor else pd.DataFrame()

print('Predicciones del mejor modelo por zona:')
display(predicciones_mejor_modelo.head())

print('\nEventos MHW detectados desde SST predicha por zona:')
if len(eventos_predichos_mejor) > 0:
    display(eventos_predichos_mejor)
else:
    print('No se detectaron eventos MHW predichos en el periodo de prueba con los mejores modelos.')

In [ ]:
# Serie observada vs predicha para el periodo de prueba
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=False)

for ax, zona in zip(axes, ['Norte', 'Centro', 'Sur']):
    sub = predicciones_mejor_modelo[predicciones_mejor_modelo['zona'] == zona]
    ax.plot(sub['fecha'], sub['sst_observada'], label='SST observada', color='black', linewidth=1)
    ax.plot(sub['fecha'], sub['sst_predicha'], label='SST predicha', color='tomato', linewidth=1)
    ax.plot(sub['fecha'], sub['umbral_p90'], label='Umbral P90', color='green', linestyle='--', linewidth=1)
    ax.fill_between(sub['fecha'], sub['sst_predicha'], sub['umbral_p90'],
                    where=sub['mhw_predicha'].astype(bool), color='orange', alpha=0.3,
                    label='MHW derivada de SST predicha')
    ax.set_title(f'SST observada vs predicha y MHW derivada - Zona {zona}')
    ax.set_ylabel('SST (C)')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Fecha')
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion: MHW observada vs MHW derivada desde SST predicha
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, zona in zip(axes, ['Norte', 'Centro', 'Sur']):
    sub = predicciones_mejor_modelo[predicciones_mejor_modelo['zona'] == zona]
    ConfusionMatrixDisplay.from_predictions(
        sub['mhw_observada'].astype(int),
        sub['mhw_predicha'].astype(int),
        labels=[0, 1], display_labels=['No MHW', 'MHW'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'Zona {zona}')

plt.suptitle('Comparacion de MHW observada vs MHW derivada de SST predicha', y=1.05)
plt.tight_layout()
plt.show()

## 13. Identificacion final de olas de calor marinas observadas

Esta parte mantiene la identificacion fisica con la SST observada. Sirve como referencia para comparar contra la deteccion obtenida desde la SST predicha.

In [ ]:
# Tabla final de eventos observados por zona
columnas_eventos = [
    'zona', 'evento_id_zona', 'fecha_inicio', 'fecha_fin', 'duracion_dias',
    'fecha_pico', 'sst_pico', 'intensidad_maxima', 'intensidad_media',
    'intensidad_acumulada', 'categoria_intensidad', 'categoria_duracion'
]

eventos_final = eventos_mhw[columnas_eventos].sort_values(['zona', 'fecha_inicio']).copy()
display(eventos_final)

print('Numero de eventos observados por zona:')
print(eventos_final.groupby('zona').size().sort_index())

In [ ]:
# Grafico de eventos observados en Peru_total, comparable con el avance inicial del proyecto
zona_eventos = 'Peru_total'
df_plot = eventos_mhw[eventos_mhw['zona'] == zona_eventos].copy().sort_values('fecha_inicio')

mapa_colores = {
    'I - Baja (<25 C dia)': '#6baed6',
    'II - Moderada (25-99 C dia)': '#74c476',
    'III - Alta (100-199 C dia)': '#fd8d3c',
    'IV - Extrema (>=200 C dia)': '#de2d26',
}

if len(df_plot) > 0:
    df_plot['etiqueta_evento'] = (
        df_plot['fecha_inicio'].dt.strftime('%Y-%m-%d') + '\na\n' +
        df_plot['fecha_fin'].dt.strftime('%Y-%m-%d')
    )
    colores = df_plot['categoria_intensidad'].map(mapa_colores).fillna('gray')

    plt.figure(figsize=(18, 7))
    plt.bar(df_plot['etiqueta_evento'], df_plot['intensidad_acumulada'], color=colores)
    plt.axhline(25, linestyle='--', linewidth=1, color='black')
    plt.axhline(100, linestyle='--', linewidth=1, color='black')
    plt.axhline(200, linestyle='--', linewidth=1, color='black')
    plt.title('Eventos MHW observados en Peru_total segun intensidad acumulada')
    plt.xlabel('Evento MHW')
    plt.ylabel('Intensidad acumulada (C dia)')
    plt.xticks(rotation=90, fontsize=7)
    leyenda = [mpatches.Patch(color=v, label=k) for k, v in mapa_colores.items()]
    plt.legend(handles=leyenda, title='Categoria', loc='upper right')
    plt.tight_layout()
    plt.show()
else:
    print('No hay eventos para Peru_total con los parametros actuales.')

#PREDICCIÓN DE PRÓXIMOS 3 MESES

In [ ]:
# ============================================================
# CELDA 1. CARGAR SST OBSERVADA 2026 Y DEFINIR HORIZONTE
# ============================================================

# Número de días a comparar:
# 14  = dos semanas
# 30  = un mes aprox.
# 90  = tres meses aprox.
# 180 = enero-junio aprox., si tienes archivos hasta junio
N_DIAS_COMPARACION = 90

# Buscar archivos 2026 en la carpeta NOAA
archivos_todos = sorted(glob.glob(str(RUTA_NOAA / "*.nc")))

archivos_2026 = sorted([
    f for f in archivos_todos
    if "2026" in os.path.basename(f)
])

print("Archivos 2026 encontrados:", len(archivos_2026))
for f in archivos_2026:
    print(os.path.basename(f))

if len(archivos_2026) == 0:
    raise FileNotFoundError(
        "No se encontraron archivos 2026. Revisa RUTA_NOAA o el nombre de los archivos."
    )

# Leer SST observada 2026 por zonas
series_2026 = []

for f in tqdm(archivos_2026, desc="Leyendo NOAA OISST 2026"):
    series_2026.append(leer_promedios_zonales_nc(f))

df_sst_2026 = (
    pd.concat(series_2026, ignore_index=True)
    .groupby(["fecha", "zona"], as_index=False)["sst_promedio"].mean()
    .sort_values(["zona", "fecha"])
    .reset_index(drop=True)
)

# Mantener solo zonas del modelo
df_sst_2026 = df_sst_2026[
    df_sst_2026["zona"].isin(["Norte", "Centro", "Sur"])
].copy()

# Filtrar horizonte de comparación
fecha_inicio = df_sst_2026["fecha"].min()
fecha_fin = fecha_inicio + pd.Timedelta(days=N_DIAS_COMPARACION - 1)

df_sst_2026 = df_sst_2026[
    (df_sst_2026["fecha"] >= fecha_inicio) &
    (df_sst_2026["fecha"] <= fecha_fin)
].copy()

print("Periodo usado para comparación:")
print(df_sst_2026["fecha"].min(), "a", df_sst_2026["fecha"].max())
print("Número de días configurado:", N_DIAS_COMPARACION)

display(df_sst_2026.head())

In [ ]:
# ============================================================
# CELDA 2. PREDECIR SST 2026 Y CREAR TABLA COMPARATIVA
# ============================================================

def construir_features_prediccion(hist_zona, fecha_pred, feature_cols):
    """
    Construye variables predictoras para una fecha futura usando el historial
    disponible hasta el día anterior.
    """

    hist_zona = hist_zona.copy()

    if "doy" not in hist_zona.columns:
        hist_zona["doy"] = doy_sin_bisiesto(hist_zona["fecha"])
        hist_zona = hist_zona.dropna(subset=["doy"]).copy()
        hist_zona["doy"] = hist_zona["doy"].astype(int)

    fila = {}

    doy_pred = doy_sin_bisiesto(pd.Series([fecha_pred])).iloc[0]
    doy_pred = int(doy_pred) if not pd.isna(doy_pred) else hist_zona["doy"].iloc[-1]

    fila["mes"] = fecha_pred.month
    fila["doy_sin"] = np.sin(2 * np.pi * doy_pred / 365)
    fila["doy_cos"] = np.cos(2 * np.pi * doy_pred / 365)

    # Climatología y umbral P90 histórico para ese día del año
    clim_ref = hist_zona[hist_zona["doy"] == doy_pred][
        ["climatologia", "umbral_p90"]
    ].drop_duplicates()

    if len(clim_ref) > 0:
        fila["climatologia"] = clim_ref["climatologia"].iloc[-1]
        fila["umbral_p90"] = clim_ref["umbral_p90"].iloc[-1]
    else:
        fila["climatologia"] = hist_zona["climatologia"].iloc[-1]
        fila["umbral_p90"] = hist_zona["umbral_p90"].iloc[-1]

    # Rezagos de SST, anomalía y exceso térmico
    for lag in [1, 3, 7, 14, 30]:
        fila[f"sst_lag{lag}"] = hist_zona["sst_promedio"].iloc[-lag]
        fila[f"anom_lag{lag}"] = hist_zona["anomalia_climatologica"].iloc[-lag]
        fila[f"exceso_lag{lag}"] = hist_zona["exceso_umbral"].iloc[-lag]

    # Medias móviles previas
    fila["sst_ma7_prev"] = hist_zona["sst_promedio"].tail(7).mean()
    fila["sst_ma30_prev"] = hist_zona["sst_promedio"].tail(30).mean()
    fila["anom_ma7_prev"] = hist_zona["anomalia_climatologica"].tail(7).mean()
    fila["exceso_ma7_prev"] = hist_zona["exceso_umbral"].tail(7).mean()

    # ICEN rezagado: se mantiene el último valor disponible como escenario persistente
    if "icen_lag1" in feature_cols:
        fila["icen_lag1"] = (
            hist_zona["icen_lag1"].dropna().iloc[-1]
            if "icen_lag1" in hist_zona.columns and hist_zona["icen_lag1"].notna().any()
            else 0
        )

    # Viento: se mantiene el último valor disponible si esas variables existen
    for col in [
        "u10_lag1", "u10_lag7",
        "v10_lag1", "v10_lag7",
        "velocidad_viento_lag1", "velocidad_viento_lag7"
    ]:
        if col in feature_cols:
            fila[col] = (
                hist_zona[col].dropna().iloc[-1]
                if col in hist_zona.columns and hist_zona[col].notna().any()
                else 0
            )

    return pd.DataFrame([{col: fila.get(col, 0) for col in feature_cols}])


# Predicción 2026 por zona usando el mejor modelo seleccionado
predicciones_2026 = []

for _, row in mejores_por_zona.iterrows():
    zona = row["zona"]

    if zona not in ["Norte", "Centro", "Sur"]:
        continue

    modelo_nombre = row["modelo"]
    modelo = modelos_por_zona[zona][modelo_nombre]

    hist_zona = df_ml[df_ml["zona"] == zona].sort_values("fecha").copy()

    if "doy" not in hist_zona.columns:
        hist_zona["doy"] = doy_sin_bisiesto(hist_zona["fecha"])
        hist_zona = hist_zona.dropna(subset=["doy"]).copy()
        hist_zona["doy"] = hist_zona["doy"].astype(int)

    fechas_2026 = sorted(
        df_sst_2026[df_sst_2026["zona"] == zona]["fecha"].unique()
    )

    registros = []

    for fecha_pred in fechas_2026:
        fecha_pred = pd.to_datetime(fecha_pred)

        X_pred = construir_features_prediccion(hist_zona, fecha_pred, feature_cols)
        sst_pred = modelo.predict(X_pred)[0]

        doy_pred = doy_sin_bisiesto(pd.Series([fecha_pred])).iloc[0]
        doy_pred = int(doy_pred) if not pd.isna(doy_pred) else hist_zona["doy"].iloc[-1]

        clim_ref = hist_zona[hist_zona["doy"] == doy_pred][
            ["climatologia", "umbral_p90"]
        ].drop_duplicates()

        if len(clim_ref) > 0:
            climatologia = clim_ref["climatologia"].iloc[-1]
            umbral_p90 = clim_ref["umbral_p90"].iloc[-1]
        else:
            climatologia = hist_zona["climatologia"].iloc[-1]
            umbral_p90 = hist_zona["umbral_p90"].iloc[-1]

        nuevo_pred = {
            "fecha": fecha_pred,
            "zona": zona,
            "modelo": modelo_nombre,
            "sst_predicha": sst_pred,
            "climatologia": climatologia,
            "umbral_p90": umbral_p90,
            "anomalia_climatologica": sst_pred - climatologia,
            "exceso_umbral": sst_pred - umbral_p90,
            "doy": doy_pred
        }

        registros.append(nuevo_pred)

        # Se agrega la predicción al historial para construir lags del siguiente día
        nuevo_hist = {
            "fecha": fecha_pred,
            "zona": zona,
            "sst_promedio": sst_pred,
            "climatologia": climatologia,
            "umbral_p90": umbral_p90,
            "anomalia_climatologica": sst_pred - climatologia,
            "exceso_umbral": sst_pred - umbral_p90,
            "doy": doy_pred
        }

        hist_zona = pd.concat(
            [hist_zona, pd.DataFrame([nuevo_hist])],
            ignore_index=True
        )

    predicciones_2026.append(pd.DataFrame(registros))

pred_sst_2026 = pd.concat(predicciones_2026, ignore_index=True)


# Preparar SST observada con climatología y umbral P90 histórico
obs_tmp = df_sst_2026.rename(columns={
    "sst_promedio": "sst_observada"
}).copy()

obs_tmp["doy"] = doy_sin_bisiesto(obs_tmp["fecha"])
obs_tmp = obs_tmp.dropna(subset=["doy"]).copy()
obs_tmp["doy"] = obs_tmp["doy"].astype(int)

obs_tmp = obs_tmp.merge(
    climatologia_zonas[["zona", "doy", "climatologia", "umbral_p90"]],
    on=["zona", "doy"],
    how="left"
)

# Tabla final observado vs predicho
comparacion_2026 = obs_tmp.merge(
    pred_sst_2026[["fecha", "zona", "modelo", "sst_predicha"]],
    on=["fecha", "zona"],
    how="inner"
)

comparacion_2026["anomalia_observada"] = (
    comparacion_2026["sst_observada"] - comparacion_2026["climatologia"]
)

comparacion_2026["anomalia_predicha"] = (
    comparacion_2026["sst_predicha"] - comparacion_2026["climatologia"]
)

comparacion_2026["exceso_umbral_observado"] = (
    comparacion_2026["sst_observada"] - comparacion_2026["umbral_p90"]
)

comparacion_2026["exceso_umbral_predicho"] = (
    comparacion_2026["sst_predicha"] - comparacion_2026["umbral_p90"]
)

comparacion_2026["error_sst"] = (
    comparacion_2026["sst_predicha"] - comparacion_2026["sst_observada"]
)

display(comparacion_2026.head())

print("Filas comparadas:", len(comparacion_2026))
print("Rango:", comparacion_2026["fecha"].min(), "a", comparacion_2026["fecha"].max())

In [ ]:
# ============================================================
# CELDA 3. IDENTIFICAR MHW OBSERVADAS/PREDICHAS Y GRAFICAR
# ============================================================

mhw_obs_2026_lista = []
mhw_pred_2026_lista = []
eventos_obs_2026_lista = []
eventos_pred_2026_lista = []

for zona in ["Norte", "Centro", "Sur"]:
    sub = comparacion_2026[comparacion_2026["zona"] == zona].copy()
    sub = sub.sort_values("fecha").reset_index(drop=True)

    # -----------------------------
    # Serie observada para Hobday
    # -----------------------------
    obs = pd.DataFrame({
        "fecha": sub["fecha"],
        "zona": zona,
        "sst_promedio": sub["sst_observada"],
        "climatologia": sub["climatologia"],
        "umbral_p90": sub["umbral_p90"]
    })

    obs["anomalia_climatologica"] = obs["sst_promedio"] - obs["climatologia"]
    obs["exceso_umbral"] = obs["sst_promedio"] - obs["umbral_p90"]
    obs["sobre_umbral"] = obs["sst_promedio"] > obs["umbral_p90"]

    diario_obs, eventos_obs = detectar_eventos_mhw_zona(
        obs,
        zona=zona,
        min_duracion=DURACION_MINIMA,
        max_gap=MAX_GAP
    )

    diario_obs = diario_obs[["fecha", "zona", "mhw"]].rename(
        columns={"mhw": "mhw_observada_hobday"}
    )

    mhw_obs_2026_lista.append(diario_obs)

    if len(eventos_obs) > 0:
        eventos_obs["tipo"] = "Observada"
        eventos_obs_2026_lista.append(eventos_obs)

    # -----------------------------
    # Serie predicha para Hobday
    # -----------------------------
    pred = pd.DataFrame({
        "fecha": sub["fecha"],
        "zona": zona,
        "sst_promedio": sub["sst_predicha"],
        "climatologia": sub["climatologia"],
        "umbral_p90": sub["umbral_p90"]
    })

    pred["anomalia_climatologica"] = pred["sst_promedio"] - pred["climatologia"]
    pred["exceso_umbral"] = pred["sst_promedio"] - pred["umbral_p90"]
    pred["sobre_umbral"] = pred["sst_promedio"] > pred["umbral_p90"]

    diario_pred, eventos_pred = detectar_eventos_mhw_zona(
        pred,
        zona=zona,
        min_duracion=DURACION_MINIMA,
        max_gap=MAX_GAP
    )

    diario_pred = diario_pred[["fecha", "zona", "mhw"]].rename(
        columns={"mhw": "mhw_predicha_hobday"}
    )

    mhw_pred_2026_lista.append(diario_pred)

    if len(eventos_pred) > 0:
        eventos_pred["tipo"] = "Predicha"
        eventos_pred_2026_lista.append(eventos_pred)


# Tablas de días MHW y eventos
mhw_obs_2026_hobday = pd.concat(mhw_obs_2026_lista, ignore_index=True)
mhw_pred_2026_hobday = pd.concat(mhw_pred_2026_lista, ignore_index=True)

eventos_obs_2026_hobday = (
    pd.concat(eventos_obs_2026_lista, ignore_index=True)
    if eventos_obs_2026_lista else pd.DataFrame()
)

eventos_pred_2026_hobday = (
    pd.concat(eventos_pred_2026_lista, ignore_index=True)
    if eventos_pred_2026_lista else pd.DataFrame()
)

# Unir identificación MHW a la tabla comparativa
comparacion_2026_mhw = comparacion_2026.merge(
    mhw_obs_2026_hobday,
    on=["fecha", "zona"],
    how="left"
).merge(
    mhw_pred_2026_hobday,
    on=["fecha", "zona"],
    how="left"
)

comparacion_2026_mhw["mhw_observada_hobday"] = (
    comparacion_2026_mhw["mhw_observada_hobday"].fillna(False).astype(bool)
)

comparacion_2026_mhw["mhw_predicha_hobday"] = (
    comparacion_2026_mhw["mhw_predicha_hobday"].fillna(False).astype(bool)
)

print("Eventos MHW observados 2026:")
display(eventos_obs_2026_hobday)

print("Eventos MHW predichos 2026:")
display(eventos_pred_2026_hobday)


# Gráfico final: SST observada, SST predicha, umbral P90 y MHW Hobday
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for ax, zona in zip(axes, ["Norte", "Centro", "Sur"]):
    sub = comparacion_2026_mhw[
        comparacion_2026_mhw["zona"] == zona
    ].copy()

    ax.plot(
        sub["fecha"],
        sub["sst_observada"],
        label="SST observada",
        color="black",
        linewidth=2
    )

    ax.plot(
        sub["fecha"],
        sub["sst_predicha"],
        label="SST predicha",
        color="tomato",
        linewidth=2,
        linestyle="--"
    )

    ax.plot(
        sub["fecha"],
        sub["umbral_p90"],
        label="Umbral P90",
        color="green",
        linewidth=1.5,
        linestyle="--"
    )

    ax.fill_between(
        sub["fecha"],
        sub["sst_observada"],
        sub["umbral_p90"],
        where=sub["mhw_observada_hobday"],
        color="red",
        alpha=0.25,
        label="MHW observada"
    )

    ax.fill_between(
        sub["fecha"],
        sub["sst_predicha"],
        sub["umbral_p90"],
        where=sub["mhw_predicha_hobday"],
        color="orange",
        alpha=0.35,
        label="MHW predicha"
    )

    ax.set_title(f"SST observada vs predicha y MHW 2026 - Zona {zona}")
    ax.set_ylabel("SST (°C)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DASHBOARD INTEGRAL DEL PROYECTO PARA GITHUB PAGES
# Genera: docs/dashboard.html y docs/index.html
# Requiere ejecutar antes el notebook hasta tener:
# comparacion_2026_mhw, tabla_modelos, mejores_por_zona
# ============================================================

# En Colab, si falta alguna librería, ejecutar una vez:
# !pip install folium plotly -q

from pathlib import Path
import numpy as np
import pandas as pd
import folium
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. Configuración editable
# ------------------------------------------------------------

TITULO_DASHBOARD = "Predicción de temperatura superficial del mar e identificación de olas de calor marinas en la costa del Perú (1990-2025)"
AUTORES = "Lucila Carolina Condezo Inocencio, Lizett Avelina Alejo Huamani, Nicoll Yrene Aguilar Quispe y María Fernanda Velarde Salazar"
CURSO = "Ciencia de Datos Ambientales - UTEC"
FUENTES = "NOAA OISST v2.1; ICEN; ERA5-Land/ERA5; procesamiento propio en Python"
ZONAS_DASH = ["Norte", "Centro", "Sur"]

# Cambia este texto si tu equipo quiere especificar nombres.
NOTA_AUTORIA = "Dashboard desarrollado en Python con Folium, Plotly y Pandas."

# ------------------------------------------------------------
# 2. Validaciones y copias de trabajo
# ------------------------------------------------------------

variables_necesarias = [
    "comparacion_2026_mhw",
    "tabla_modelos",
    "mejores_por_zona",
]

faltantes = [v for v in variables_necesarias if v not in globals()]
if faltantes:
    raise NameError(
        "Faltan variables para construir el dashboard: " + ", ".join(faltantes) +
        ". Ejecuta antes las celdas de modelamiento, comparación 2026 y detección MHW."
    )

comp = comparacion_2026_mhw.copy()
comp["fecha"] = pd.to_datetime(comp["fecha"])
comp = comp[comp["zona"].isin(ZONAS_DASH)].copy()

# Asegurar flags booleanos
for col in ["mhw_observada_hobday", "mhw_predicha_hobday"]:
    if col in comp.columns:
        comp[col] = comp[col].fillna(False).astype(bool)

# ------------------------------------------------------------
# 3. Funciones auxiliares
# ------------------------------------------------------------

def tabla_html(df, columnas=None, max_rows=40, decimales=4):
    """Convierte DataFrame en tabla HTML con formato."""
    if df is None or len(df) == 0:
        return "<p class='nota'><em>No se registraron datos para esta tabla.</em></p>"

    tmp = df.copy()
    if columnas is not None:
        cols = [c for c in columnas if c in tmp.columns]
        tmp = tmp[cols]

    for col in tmp.columns:
        if np.issubdtype(tmp[col].dtype, np.datetime64):
            tmp[col] = pd.to_datetime(tmp[col]).dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_numeric_dtype(tmp[col]):
            tmp[col] = tmp[col].round(decimales)

    return tmp.head(max_rows).to_html(index=False, classes="tabla", border=0, escape=False)


def calcular_r2(obs, pred):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    ss_res = np.sum((pred - obs) ** 2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan


def metricas_sst_por_grupo(g):
    obs = g["sst_observada"].astype(float).to_numpy()
    pred = g["sst_predicha"].astype(float).to_numpy()
    err = pred - obs
    return pd.Series({
        "MAE_SST": np.mean(np.abs(err)),
        "RMSE_SST": np.sqrt(np.mean(err ** 2)),
        "R2_SST": calcular_r2(obs, pred),
        "Bias_SST": np.mean(err),
        "dias_MHW_observados": int(g["mhw_observada_hobday"].sum()),
        "dias_MHW_predichos": int(g["mhw_predicha_hobday"].sum()),
    })


def intervalos_flag(df, col_flag):
    """Devuelve intervalos continuos donde col_flag=True."""
    df = df.sort_values("fecha").reset_index(drop=True)
    intervalos = []
    inicio = None
    ultimo = None

    for fecha, flag in zip(df["fecha"], df[col_flag].astype(bool)):
        if flag and inicio is None:
            inicio = fecha
        if flag:
            ultimo = fecha
        if (not flag) and inicio is not None:
            intervalos.append((inicio, ultimo))
            inicio = None
            ultimo = None

    if inicio is not None:
        intervalos.append((inicio, ultimo))
    return intervalos


def safe_len(varname):
    return len(globals()[varname]) if varname in globals() and globals()[varname] is not None else 0

# ------------------------------------------------------------
# 4. Métricas, tablas y resúmenes automáticos
# ------------------------------------------------------------

fecha_comp_min = comp["fecha"].min()
fecha_comp_max = comp["fecha"].max()
n_dias_comp = comp["fecha"].nunique()

metricas_2026 = (
    comp.groupby("zona")
    .apply(metricas_sst_por_grupo)
    .reset_index()
)

# Tabla de desempeño de modelos durante entrenamiento/prueba histórica
tabla_perf = tabla_modelos.copy()
tabla_perf = tabla_perf[tabla_perf["zona"].isin(ZONAS_DASH)].copy()

cols_perf = ["zona", "modelo", "mae_sst", "rmse_sst", "r2_sst", "bias_sst",
             "precision_mhw_derivada", "recall_mhw_derivada", "f1_mhw_derivada"]
cols_perf = [c for c in cols_perf if c in tabla_perf.columns]

# Mejor modelo por zona
mejores_dash = mejores_por_zona.copy()
mejores_dash = mejores_dash[mejores_dash["zona"].isin(ZONAS_DASH)].copy()

# Resumen histórico opcional
n_eventos_hist = safe_len("eventos_mhw")
if n_eventos_hist == 0 and "eventos_mhw_clasificados" in globals():
    n_eventos_hist = len(eventos_mhw_clasificados)

# Estadísticos EDA por zona, si existe df_sst_zonas
if "df_sst_zonas" in globals():
    eda_zonal = df_sst_zonas[df_sst_zonas["zona"].isin(ZONAS_DASH)].copy()
    eda_zonal["fecha"] = pd.to_datetime(eda_zonal["fecha"])
    tabla_eda = (
        eda_zonal.groupby("zona")["sst_promedio"]
        .agg(["count", "mean", "std", "min", "max"])
        .reset_index()
        .rename(columns={
            "count": "n_registros",
            "mean": "sst_media",
            "std": "sst_desv_est",
            "min": "sst_min",
            "max": "sst_max"
        })
    )
    periodo_hist_min = eda_zonal["fecha"].min().strftime("%Y-%m-%d")
    periodo_hist_max = eda_zonal["fecha"].max().strftime("%Y-%m-%d")
    faltantes_sst = int(eda_zonal["sst_promedio"].isna().sum())
else:
    tabla_eda = pd.DataFrame()
    periodo_hist_min = "N.D."
    periodo_hist_max = "N.D."
    faltantes_sst = "N.D."

# Días MHW anuales, si existe mhw_diario
if "mhw_diario" in globals():
    mhw_tmp = mhw_diario.copy()
    mhw_tmp["fecha"] = pd.to_datetime(mhw_tmp["fecha"])
    if "mhw" in mhw_tmp.columns:
        dias_mhw_anuales = (
            mhw_tmp[mhw_tmp["zona"].isin(ZONAS_DASH)]
            .assign(anio=lambda x: x["fecha"].dt.year)
            .groupby(["anio", "zona"], as_index=False)["mhw"]
            .sum()
            .rename(columns={"mhw": "dias_mhw"})
        )
    else:
        dias_mhw_anuales = pd.DataFrame()
else:
    dias_mhw_anuales = pd.DataFrame()

# Eventos 2026 observados/predichos, si existen
if "eventos_obs_2026_hobday" in globals():
    eventos_obs_dash = eventos_obs_2026_hobday.copy()
else:
    eventos_obs_dash = pd.DataFrame()

if "eventos_pred_2026_hobday" in globals():
    eventos_pred_dash = eventos_pred_2026_hobday.copy()
else:
    eventos_pred_dash = pd.DataFrame()

n_eventos_obs_2026 = len(eventos_obs_dash)
n_eventos_pred_2026 = len(eventos_pred_dash)

# Mejor desempeño global por RMSE 2026
if len(metricas_2026) > 0:
    mejor_zona_2026 = metricas_2026.sort_values("RMSE_SST").iloc[0]
    mejor_zona_txt = str(mejor_zona_2026["zona"])
    mejor_rmse_txt = f"{mejor_zona_2026['RMSE_SST']:.3f} °C"
else:
    mejor_zona_txt = "N.D."
    mejor_rmse_txt = "N.D."

# ------------------------------------------------------------
# 5. Mapa Folium del área de estudio
# ------------------------------------------------------------

mapa = folium.Map(location=[-10.0, -76.5], zoom_start=5, tiles="CartoDB positron")

# Cajas aproximadas de análisis zonal. Ajustar si tu recorte usa otros límites.
zonas_geo = {
    "Norte":  {"lat_min": -6.0,  "lat_max": 0.0,   "lon_min": -84.5, "lon_max": -77.0},
    "Centro": {"lat_min": -13.0, "lat_max": -6.0,  "lon_min": -83.5, "lon_max": -75.0},
    "Sur":    {"lat_min": -20.0, "lat_max": -13.0, "lon_min": -82.0, "lon_max": -70.0},
}

colores_zona = {"Norte": "red", "Centro": "orange", "Sur": "blue"}

for zona, lim in zonas_geo.items():
    folium.Rectangle(
        bounds=[[lim["lat_min"], lim["lon_min"]], [lim["lat_max"], lim["lon_max"]]],
        color=colores_zona[zona],
        fill=True,
        fill_opacity=0.16,
        weight=2,
        popup=(
            f"<b>Zona {zona}</b><br>"
            f"Latitud aprox.: {lim['lat_min']} a {lim['lat_max']}<br>"
            f"Longitud aprox.: {lim['lon_min']} a {lim['lon_max']}"
        ),
        tooltip=f"Zona {zona}"
    ).add_to(mapa)

folium.Marker(
    location=[-12.05, -77.05],
    tooltip="Costa peruana",
    popup="Área de análisis: costa peruana, NOAA OISST procesado por zonas."
).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa_html = mapa.get_root().render()

# ------------------------------------------------------------
# 6. Gráfico Plotly principal: SST observada vs predicha + MHW
# ------------------------------------------------------------

fig_sst = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Zona {z}" for z in ZONAS_DASH],
    vertical_spacing=0.08
)

for i, zona in enumerate(ZONAS_DASH, start=1):
    sub = comp[comp["zona"] == zona].sort_values("fecha").copy()

    fig_sst.add_trace(
        go.Scatter(
            x=sub["fecha"],
            y=sub["sst_observada"],
            mode="lines",
            name="SST observada" if i == 1 else None,
            line=dict(color="black", width=2),
            showlegend=(i == 1),
            hovertemplate="%{x|%Y-%m-%d}<br>SST obs: %{y:.2f} °C<extra></extra>"
        ), row=i, col=1
    )

    fig_sst.add_trace(
        go.Scatter(
            x=sub["fecha"],
            y=sub["sst_predicha"],
            mode="lines",
            name="SST predicha" if i == 1 else None,
            line=dict(color="tomato", width=2, dash="dash"),
            showlegend=(i == 1),
            hovertemplate="%{x|%Y-%m-%d}<br>SST pred: %{y:.2f} °C<extra></extra>"
        ), row=i, col=1
    )

    fig_sst.add_trace(
        go.Scatter(
            x=sub["fecha"],
            y=sub["umbral_p90"],
            mode="lines",
            name="Umbral P90" if i == 1 else None,
            line=dict(color="green", width=1.5, dash="dot"),
            showlegend=(i == 1),
            hovertemplate="%{x|%Y-%m-%d}<br>P90: %{y:.2f} °C<extra></extra>"
        ), row=i, col=1
    )

    for x0, x1 in intervalos_flag(sub, "mhw_observada_hobday"):
        fig_sst.add_vrect(
            x0=x0,
            x1=x1 + pd.Timedelta(days=1),
            fillcolor="red",
            opacity=0.12,
            line_width=0,
            row=i,
            col=1
        )

    for x0, x1 in intervalos_flag(sub, "mhw_predicha_hobday"):
        fig_sst.add_vrect(
            x0=x0,
            x1=x1 + pd.Timedelta(days=1),
            fillcolor="orange",
            opacity=0.18,
            line_width=0,
            row=i,
            col=1
        )

    fig_sst.update_yaxes(title_text="SST (°C)", row=i, col=1)

fig_sst.update_layout(
    height=920,
    title="SST observada vs SST predicha, umbral P90 y eventos MHW identificados con Hobday",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=50, r=30, t=90, b=50)
)

fig_sst_html = pio.to_html(fig_sst, full_html=False, include_plotlyjs="cdn")

# ------------------------------------------------------------
# 7. Gráficos Plotly secundarios
# ------------------------------------------------------------

# 7.1 Desempeño de modelos según RMSE
if len(tabla_perf) > 0 and "rmse_sst" in tabla_perf.columns:
    fig_modelos = px.bar(
        tabla_perf,
        x="zona",
        y="rmse_sst",
        color="modelo",
        barmode="group",
        title="Comparación de configuraciones Random Forest según RMSE",
        labels={"zona": "Zona", "rmse_sst": "RMSE de SST (°C)", "modelo": "Configuración"}
    )
    fig_modelos.update_layout(height=480, margin=dict(l=50, r=30, t=70, b=50))
    fig_modelos_html = pio.to_html(fig_modelos, full_html=False, include_plotlyjs=False)
else:
    fig_modelos_html = "<p class='nota'>No se encontró tabla de desempeño de modelos.</p>"

# 7.2 Días MHW anuales históricos
if len(dias_mhw_anuales) > 0:
    fig_mhw_anual = px.line(
        dias_mhw_anuales,
        x="anio",
        y="dias_mhw",
        color="zona",
        markers=True,
        title="Días bajo condición MHW por año y zona",
        labels={"anio": "Año", "dias_mhw": "Días MHW", "zona": "Zona"}
    )
    fig_mhw_anual.update_layout(height=470, margin=dict(l=50, r=30, t=70, b=50))
    fig_mhw_anual_html = pio.to_html(fig_mhw_anual, full_html=False, include_plotlyjs=False)
else:
    fig_mhw_anual_html = "<p class='nota'>No se encontró la serie diaria MHW histórica para graficar días MHW anuales.</p>"

# ------------------------------------------------------------
# 8. Tablas HTML
# ------------------------------------------------------------

tabla_indicadores = pd.DataFrame([
    {"Indicador": "SST promedio zonal", "Unidad": "°C", "Uso": "Variable objetivo del modelo y base de detección MHW."},
    {"Indicador": "Anomalía climatológica de SST", "Unidad": "°C", "Uso": "Diferencia entre SST y climatología diaria."},
    {"Indicador": "Umbral P90", "Unidad": "°C", "Uso": "Referencia extrema diaria para identificar días candidatos MHW."},
    {"Indicador": "Días MHW", "Unidad": "días", "Uso": "Frecuencia de condiciones de ola de calor marina."},
    {"Indicador": "Duración e intensidad acumulada", "Unidad": "días; °C día", "Uso": "Persistencia y severidad de cada evento."},
])

tabla_fuentes = pd.DataFrame([
    {"Fuente": "NOAA OISST", "Formato": "NetCDF (.nc)", "Variable": "SST diaria", "Uso": "Variable principal para climatología, modelo y MHW."},
    {"Fuente": "ICEN", "Formato": "Excel/CSV", "Variable": "Índice Costero El Niño", "Uso": "Variable climática complementaria rezagada."},
    {"Fuente": "ERA5/ERA5-Land", "Formato": "NetCDF (.nc)", "Variable": "Viento u10/v10", "Uso": "Variable atmosférica complementaria si está disponible."},
])

tabla_pipeline = pd.DataFrame([
    {"Etapa": "1. Ingesta", "Descripción": "Carga de archivos NetCDF NOAA OISST y datos complementarios."},
    {"Etapa": "2. Preparación", "Descripción": "Validación temporal, limpieza, promedio espacial y división Norte/Centro/Sur."},
    {"Etapa": "3. EDA", "Descripción": "Revisión de valores faltantes, estadísticos, anomalías, patrones temporales y MHW históricas."},
    {"Etapa": "4. Modelamiento", "Descripción": "Random Forest Regressor con tres configuraciones y división cronológica 70/30."},
    {"Etapa": "5. Evaluación", "Descripción": "MAE, RMSE, R² para SST y métricas derivadas de MHW."},
    {"Etapa": "6. Visualización", "Descripción": "Dashboard HTML con mapa, gráficos interactivos y tablas de resultados."},
])

indicadores_html = tabla_html(tabla_indicadores, decimales=3)
fuentes_html = tabla_html(tabla_fuentes, decimales=3)
pipeline_html = tabla_html(tabla_pipeline, decimales=3)
eda_html = tabla_html(tabla_eda, decimales=3)
modelos_html = tabla_html(tabla_perf, columnas=cols_perf, max_rows=20, decimales=4)
mejores_html = tabla_html(
    mejores_dash,
    columnas=["zona", "modelo", "mae_sst", "rmse_sst", "r2_sst", "precision_mhw_derivada", "recall_mhw_derivada", "f1_mhw_derivada"],
    max_rows=10,
    decimales=4
)
metricas_2026_html = tabla_html(metricas_2026, decimales=4)
eventos_obs_html = tabla_html(
    eventos_obs_dash,
    columnas=["zona", "fecha_inicio", "fecha_fin", "duracion_dias", "intensidad_maxima", "intensidad_acumulada", "tipo"],
    max_rows=25,
    decimales=3
)
eventos_pred_html = tabla_html(
    eventos_pred_dash,
    columnas=["zona", "fecha_inicio", "fecha_fin", "duracion_dias", "intensidad_maxima", "intensidad_acumulada", "tipo"],
    max_rows=25,
    decimales=3
)

# ------------------------------------------------------------
# 9. HTML final integral
# ------------------------------------------------------------

fecha_min_txt = fecha_comp_min.strftime("%Y-%m-%d")
fecha_max_txt = fecha_comp_max.strftime("%Y-%m-%d")

html_final = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>{TITULO_DASHBOARD}</title>
    <style>
        :root {{
            --azul: #0b1f33;
            --azul2: #123a5a;
            --fondo: #f4f7fb;
            --texto: #1f2933;
            --gris: #5f6b7a;
            --borde: #d9e2ec;
        }}
        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: var(--fondo);
            color: var(--texto);
            line-height: 1.55;
        }}
        header {{
            background: linear-gradient(120deg, var(--azul), var(--azul2));
            color: white;
            padding: 34px 48px;
        }}
        header h1 {{
            margin: 0 0 10px 0;
            font-size: 32px;
        }}
        header p {{
            margin: 5px 0;
            color: #dbeafe;
        }}
        nav {{
            background: white;
            border-bottom: 1px solid var(--borde);
            padding: 12px 48px;
            position: sticky;
            top: 0;
            z-index: 999;
        }}
        nav a {{
            margin-right: 18px;
            color: var(--azul);
            text-decoration: none;
            font-size: 14px;
            font-weight: 600;
        }}
        main {{
            padding: 26px 48px 48px 48px;
        }}
        section {{
            background: white;
            margin-bottom: 24px;
            padding: 24px;
            border-radius: 14px;
            box-shadow: 0 2px 12px rgba(0, 0, 0, 0.06);
        }}
        h2 {{
            margin-top: 0;
            color: var(--azul);
            font-size: 23px;
        }}
        h3 {{
            color: var(--azul2);
            margin-top: 20px;
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));
            gap: 14px;
        }}
        .kpi {{
            background: #eef5fb;
            border-left: 5px solid var(--azul);
            padding: 16px;
            border-radius: 10px;
        }}
        .kpi .valor {{
            font-size: 24px;
            font-weight: 700;
            margin-bottom: 4px;
        }}
        .kpi .label {{
            color: var(--gris);
            font-size: 13px;
        }}
        .nota {{
            color: var(--gris);
            font-size: 14px;
        }}
        .tabla {{
            border-collapse: collapse;
            width: 100%;
            font-size: 13px;
            margin-top: 10px;
        }}
        .tabla th {{
            background: var(--azul);
            color: white;
            padding: 9px;
            text-align: left;
        }}
        .tabla td {{
            border-bottom: 1px solid var(--borde);
            padding: 8px;
            vertical-align: top;
        }}
        .two-col {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
            gap: 20px;
        }}
        ul {{
            padding-left: 22px;
        }}
        li {{
            margin-bottom: 6px;
        }}
        .callout {{
            background: #f8fafc;
            border: 1px solid var(--borde);
            padding: 14px;
            border-radius: 10px;
        }}
        footer {{
            background: var(--azul);
            color: white;
            padding: 18px 48px;
            font-size: 13px;
        }}
        @media (max-width: 800px) {{
            header, nav, main, footer {{ padding-left: 20px; padding-right: 20px; }}
            nav {{ position: static; }}
            nav a {{ display: inline-block; margin-bottom: 8px; }}
        }}
    </style>
</head>
<body>
    <header>
        <h1>{TITULO_DASHBOARD}</h1>
        <p><strong>{CURSO}</strong></p>
        <p>Predicción de temperatura superficial del mar y detección de olas de calor marinas mediante la metodología de Hobday.</p>
        <p><strong>Autores:</strong> {AUTORES}</p>
        <p><strong>Fuentes:</strong> {FUENTES}</p>
    </header>

    <nav>
        <a href="#problema">Problema</a>
        <a href="#datos">Datos</a>
        <a href="#eda">EDA</a>
        <a href="#modelo">Modelo</a>
        <a href="#resultados">Resultados</a>
        <a href="#contribuciones">Contribuciones</a>
        <a href="#conclusiones">Conclusiones</a>
    </nav>

    <main>
        <section id="resumen">
            <h2>Resumen ejecutivo</h2>
            <p>
                Este dashboard resume el análisis de olas de calor marinas frente a la costa peruana. El flujo metodológico
                separa dos etapas: primero se predice la temperatura superficial del mar (SST) con Random Forest Regressor;
                luego se aplica la metodología de Hobday sobre la SST observada y sobre la SST predicha para identificar
                eventos MHW comparables.
            </p>
            <div class="grid">
                <div class="kpi"><div class="valor">{periodo_hist_min}</div><div class="label">Inicio del periodo histórico procesado</div></div>
                <div class="kpi"><div class="valor">{periodo_hist_max}</div><div class="label">Fin del periodo histórico procesado</div></div>
                <div class="kpi"><div class="valor">{fecha_min_txt} a {fecha_max_txt}</div><div class="label">Periodo comparado observado vs predicho</div></div>
                <div class="kpi"><div class="valor">{n_eventos_hist}</div><div class="label">Eventos MHW históricos detectados</div></div>
                <div class="kpi"><div class="valor">{n_eventos_obs_2026}</div><div class="label">Eventos MHW observados en el periodo comparado</div></div>
                <div class="kpi"><div class="valor">{n_eventos_pred_2026}</div><div class="label">Eventos MHW predichos en el periodo comparado</div></div>
                <div class="kpi"><div class="valor">{mejor_zona_txt}</div><div class="label">Zona con menor RMSE en la comparación</div></div>
                <div class="kpi"><div class="valor">{mejor_rmse_txt}</div><div class="label">Mejor RMSE observado vs predicho</div></div>
            </div>
        </section>

        <section id="problema">
            <h2>1. Descripción del problema ambiental</h2>
            <p>
                Las olas de calor marinas son periodos prolongados de temperatura superficial del mar anormalmente alta.
                En la costa peruana, estos eventos pueden alterar ecosistemas marinos, afectar procesos de afloramiento,
                modificar la disponibilidad de recursos pesqueros y asociarse con condiciones oceanográficas extremas.
            </p>
            <p>
                El problema abordado es la necesidad de monitorear y anticipar condiciones térmicas extremas en la superficie
                del mar. Para ello, el proyecto estima la SST y posteriormente identifica eventos MHW a partir de un umbral
                climatológico común.
            </p>
            <h3>Indicadores ambientales clave</h3>
            {indicadores_html}
            <h3>Hipótesis de trabajo</h3>
            <ul>
                <li>La SST diaria puede estimarse a partir de su comportamiento previo, variables estacionales y variables climáticas complementarias.</li>
                <li>El desempeño del modelo puede variar entre zona Norte, Centro y Sur debido a diferencias oceanográficas regionales.</li>
                <li>La detección de MHW derivada de la SST predicha será sensible a errores pequeños cuando la temperatura esté cerca del umbral P90.</li>
            </ul>
        </section>

        <section id="datos">
            <h2>2. Fuentes de datos, arquitectura y flujo</h2>
            <p>
                La fuente principal es NOAA OISST en formato NetCDF, procesada a escala diaria para obtener la SST promedio
                por zonas de la costa peruana. Las variables complementarias, cuando están disponibles, se incorporan como
                predictores climáticos o atmosféricos.
            </p>
            <h3>Fuentes utilizadas</h3>
            {fuentes_html}
            <h3>Pipeline de procesamiento</h3>
            {pipeline_html}
            <h3>Mapa interactivo del área de estudio</h3>
            <p class="nota">Las cajas representan una aproximación de la división regional usada para comparar el desempeño del modelo.</p>
            {mapa_html}
        </section>

        <section id="eda">
            <h2>3. Análisis descriptivo</h2>
            <p>
                El análisis exploratorio incluyó revisión de continuidad temporal, valores faltantes, estadísticas por zona,
                anomalías térmicas y frecuencia de días bajo condición MHW. La tabla resume la distribución de SST por zona
                durante el periodo histórico procesado.
            </p>
            <div class="grid">
                <div class="kpi"><div class="valor">{faltantes_sst}</div><div class="label">Valores faltantes de SST detectados en la base histórica zonal</div></div>
                <div class="kpi"><div class="valor">{n_dias_comp}</div><div class="label">Días comparados en validación observada vs predicha</div></div>
            </div>
            <h3>Estadísticos de SST por zona</h3>
            {eda_html}
            <h3>Días MHW anuales por zona</h3>
            {fig_mhw_anual_html}
            <p class="nota">
                Los outliers térmicos no se eliminaron automáticamente porque pueden representar señales ambientales reales,
                precisamente relevantes para el estudio de eventos extremos. La interpretación se hizo comparando SST,
                climatología y umbral P90.
            </p>
        </section>

        <section id="metodologia">
            <h2>4. Metodología de detección de olas de calor marinas</h2>
            <div class="two-col">
                <div class="callout">
                    <h3>Criterios Hobday aplicados</h3>
                    <ul>
                        <li>Cálculo de climatología diaria.</li>
                        <li>Estimación del umbral diario del percentil 90.</li>
                        <li>Identificación de días candidatos cuando la SST supera el P90.</li>
                        <li>Definición de evento MHW cuando la condición dura al menos 5 días.</li>
                        <li>Unión de eventos separados por brechas cortas de hasta 2 días.</li>
                    </ul>
                </div>
                <div class="callout">
                    <h3>Separación metodológica clave</h3>
                    <p>
                        El modelo no predice directamente una ola de calor marina. El modelo predice SST. Después, la
                        metodología Hobday se aplica sobre la SST predicha y sobre la SST observada usando el mismo umbral P90.
                    </p>
                </div>
            </div>
        </section>

        <section id="modelo">
            <h2>5. Desarrollo del modelo predictivo</h2>
            <p>
                Se implementó Random Forest Regressor para predecir SST diaria por zona. La división de datos fue cronológica:
                el primer 70% se usó para entrenamiento y el 30% final para prueba, evitando mezclar información futura con
                información pasada. Se probaron tres configuraciones de complejidad distinta.
            </p>
            <p>
                Las variables predictoras incluyen rezagos de SST, medias móviles, anomalías previas, exceso térmico previo,
                estacionalidad anual e ICEN rezagado cuando está disponible. El ICEN se usa rezagado para evitar incorporar
                información contemporánea del mismo mes en una lógica predictiva.
            </p>
            <h3>Comparación de configuraciones Random Forest</h3>
            {fig_modelos_html}
            <h3>Métricas históricas por modelo y zona</h3>
            {modelos_html}
            <h3>Mejor modelo seleccionado por zona</h3>
            {mejores_html}
        </section>

        <section id="resultados">
            <h2>6. Resultados: SST observada vs predicha y MHW</h2>
            <p>
                El gráfico compara la SST observada y predicha durante el periodo de validación disponible. La línea verde
                corresponde al umbral P90 histórico. Las franjas rojas representan MHW observadas y las franjas naranjas
                representan MHW derivadas de la SST predicha, ambas identificadas con los criterios de Hobday.
            </p>
            {fig_sst_html}
            <h3>Métricas de comparación observado vs predicho</h3>
            {metricas_2026_html}
            <h3>Eventos MHW observados</h3>
            {eventos_obs_html}
            <h3>Eventos MHW predichos</h3>
            {eventos_pred_html}
        </section>

        <section id="contribuciones">
            <h2>7. Contribuciones esperadas</h2>
            <ul>
                <li><strong>Predicción ambiental:</strong> estima SST diaria por zona mediante un modelo reproducible de machine learning.</li>
                <li><strong>Monitoreo de eventos extremos:</strong> identifica MHW observadas y predichas bajo una metodología climática consistente.</li>
                <li><strong>Soporte a gestión ambiental:</strong> genera indicadores útiles para seguimiento de condiciones térmicas anómalas en el mar peruano.</li>
            </ul>
        </section>

        <section id="conclusiones">
            <h2>8. Conclusiones, recomendaciones y limitaciones</h2>
            <h3>Síntesis de hallazgos</h3>
            <ul>
                <li>La SST es una variable con alta persistencia temporal, por lo que los rezagos y medias móviles aportan información relevante para su predicción.</li>
                <li>La identificación de MHW depende directamente de la calidad de la predicción de SST y de su posición respecto al umbral P90.</li>
                <li>El análisis zonal permite observar diferencias regionales en desempeño y en ocurrencia de eventos extremos.</li>
                <li>El mejor desempeño de validación se obtuvo en la zona {mejor_zona_txt}, con RMSE {mejor_rmse_txt} durante el periodo comparado.</li>
            </ul>
            <h3>Recomendaciones técnicas</h3>
            <ul>
                <li>Actualizar el modelo con nuevos datos observados para reducir la acumulación de error en predicciones recursivas.</li>
                <li>Priorizar horizontes cortos de predicción cuando el objetivo sea alerta temprana operativa.</li>
                <li>Complementar la SST con variables oceanográficas y atmosféricas de mayor resolución cuando estén disponibles.</li>
                <li>Evaluar la detección MHW no solo por duración, sino también intensidad acumulada.</li>
            </ul>
            <h3>Limitaciones</h3>
            <ul>
                <li>La predicción de varios días se realiza de forma recursiva, por lo que los errores pueden acumularse.</li>
                <li>Pequeños errores de SST cerca del P90 pueden cambiar la clasificación de un día como MHW o no MHW.</li>
                <li>El promedio espacial por zona puede ocultar procesos locales de menor escala, como afloramiento costero localizado.</li>
                <li>El ICEN es mensual, por lo que funciona como variable climática de contexto y no como señal diaria.</li>
            </ul>
        </section>
    </main>

    <footer>
        {NOTA_AUTORIA} Archivo generado: docs/dashboard.html. Fuentes: {FUENTES}.
    </footer>
</body>
</html>
"""

# ------------------------------------------------------------
# 10. Guardar para GitHub Pages
# ------------------------------------------------------------

DOCS_DIR = Path("docs")
DOCS_DIR.mkdir(exist_ok=True)

ruta_dashboard = DOCS_DIR / "dashboard.html"
ruta_index = DOCS_DIR / "index.html"

ruta_dashboard.write_text(html_final, encoding="utf-8")
ruta_index.write_text(html_final, encoding="utf-8")

print("Dashboard integral generado correctamente.")
print("Archivo principal:", ruta_dashboard.resolve())
print("Archivo alternativo para URL raíz:", ruta_index.resolve())
print("Sube la carpeta docs/ a GitHub y configura Pages en main /docs.")


## 14. Interpretacion de la comparacion por zonas

Usa esta seccion para redactar los resultados luego de ejecutar el notebook:

- Si una zona obtiene menor RMSE, significa que la SST de esa zona fue predicha con menor error promedio.
- Si una zona obtiene buen RMSE pero bajo F1 MHW derivado, significa que pequenos errores de temperatura cerca del umbral P90 pueden afectar la deteccion de eventos.
- Si el recall MHW derivado es alto, la SST predicha permite recuperar buena parte de los dias MHW observados.
- Si la precision MHW derivada es baja, el metodo estaria generando falsas alarmas de MHW.
- La division Norte/Centro/Sur permite evaluar si el modelo funciona mejor en zonas con senales termicas mas marcadas o con menor variabilidad local.

## 15. Contribuciones esperadas del proyecto

1. **Prediccion ambiental:** estima la SST diaria en la costa peruana usando Random Forest Regressor y variables temporales/rezagadas.
2. **Deteccion metodologica:** aplica Hobday sobre SST observada y sobre SST predicha, separando claramente prediccion de temperatura e identificacion de MHW.
3. **Comparacion regional:** evalua diferencias entre zona Norte, Centro y Sur para identificar donde el modelo predice mejor la SST.
4. **Gestion y monitoreo:** genera una base reproducible para anticipar condiciones termicas que podrian derivar en olas de calor marinas.

## 16. Conclusiones para completar despues de ejecutar

Completar los valores numericos despues de correr todo el notebook:

1. Se identificaron **156** en el dominio `Peru_total` durante 1990-2025. Los anos con mayor numero de dias MHW fueron **2023, 1997, 1998**.
2. La comparacion zonal mostro que la zona **Norte** presento mayor frecuencia o severidad de eventos observados, segun los indicadores de dias MHW e intensidad acumulada.
3. En la prediccion de SST, el mejor desempeno se obtuvo en la zona **Sur** con el modelo **RF_3_profundo**, alcanzando un RMSE de **0.1054 °C** y un R2 de **0.9978**.
4. Al aplicar Hobday sobre la SST predicha, la deteccion MHW obtuvo un F1 de **0.9471** en la zona **Norte**, mostrando que la calidad de deteccion depende directamente del error de prediccion de temperatura.
5. Como limitacion, pequenos errores en SST cerca del umbral P90 pueden cambiar la clasificacion de un dia como MHW o no MHW. Ademas, el promedio espacial por zona puede ocultar procesos locales de menor escala.

In [ ]:
# ============================================================
# CONCLUSIONES AUTOMÁTICAS CON LOS VALORES
# ============================================================

from IPython.display import Markdown, display as ipy_display

# ------------------------------------------------------------
# 1. EVENTOS Y DÍAS MHW HISTÓRICOS
# ------------------------------------------------------------
n_eventos_hist_total = len(eventos_mhw)
n_dias_hist_total    = int(mhw_diario['mhw'].sum())

# Top 3 años con más días MHW (histórico)
top3_anios = (
    resumen_anual
    .groupby('anio')['dias_mhw']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
    .tolist()
)
top3_txt = ', '.join(str(a) for a in top3_anios)

# ------------------------------------------------------------
# 2. ZONA CON MAYOR FRECUENCIA / SEVERIDAD MHW OBSERVADA
# ------------------------------------------------------------
dias_por_zona = resumen_anual.groupby('zona')['dias_mhw'].sum()
zona_mayor_dias = dias_por_zona.idxmax()
dias_mayor      = int(dias_por_zona.max())

int_por_zona = (
    eventos_mhw
    .groupby('zona')['intensidad_acumulada']
    .sum()
    .sort_values(ascending=False)
)
zona_mayor_int = int_por_zona.idxmax()

# ------------------------------------------------------------
# 3. MEJOR MODELO POR ZONA (SST)
# ------------------------------------------------------------
# Mejor global por RMSE
mejor_global_row = tabla_modelos.sort_values('rmse_sst').iloc[0]
mejor_zona_conc  = mejor_global_row['zona']
mejor_modelo_conc = mejor_global_row['modelo']
mejor_rmse_conc  = mejor_global_row['rmse_sst']
mejor_r2_conc    = mejor_global_row['r2_sst']
mejor_mae_conc   = mejor_global_row['mae_sst']

# Tabla resumen mejores por zona
resumen_mejores = mejores_por_zona[[
    'zona', 'modelo', 'rmse_sst', 'r2_sst', 'mae_sst',
    'precision_mhw_derivada', 'recall_mhw_derivada', 'f1_mhw_derivada'
]].copy().reset_index(drop=True)

# ------------------------------------------------------------
# 4. DETECCIÓN MHW DERIVADA DE SST PREDICHA
# ------------------------------------------------------------
# Mejor F1 MHW derivada
idx_best_f1 = tabla_modelos['f1_mhw_derivada'].idxmax()
mejor_f1_row  = tabla_modelos.loc[idx_best_f1]
mejor_f1_zona = mejor_f1_row['zona']
mejor_f1_val  = mejor_f1_row['f1_mhw_derivada']
mejor_f1_mod  = mejor_f1_row['modelo']

mejor_recall    = mejor_f1_row['recall_mhw_derivada']
mejor_precision = mejor_f1_row['precision_mhw_derivada']

# ------------------------------------------------------------
# 5. EVENTOS MHW EN PERIODO DE COMPARACIÓN (si existe)
# ------------------------------------------------------------
try:
    n_ev_obs_comp  = len(eventos_obs_2026_hobday)
    n_ev_pred_comp = len(eventos_pred_2026_hobday)
    n_dias_obs_comp  = int(comparacion_2026_mhw['mhw_observada_hobday'].sum())
    n_dias_pred_comp = int(comparacion_2026_mhw['mhw_predicha_hobday'].sum())
    tiene_comparacion = True
except NameError:
    tiene_comparacion = False

# ------------------------------------------------------------
# 6. IMPRIMIR TABLA DE APOYO
# ------------------------------------------------------------
print("=" * 65)
print("  VALORES CLAVE PARA LAS CONCLUSIONES")
print("=" * 65)
print(f"\n📌 Eventos MHW históricos detectados (total): {n_eventos_hist_total}")
print(f"📌 Días MHW históricos (total 1990–2025)   : {n_dias_hist_total}")
print(f"📌 Top 3 años con más días MHW             : {top3_txt}")
print(f"\n📌 Zona con más días MHW observados        : {zona_mayor_dias} ({dias_mayor} días)")
print(f"📌 Zona con mayor intensidad acumulada     : {zona_mayor_int}")
print(f"\n📌 Mejor modelo global (menor RMSE SST)    : {mejor_modelo_conc} — Zona {mejor_zona_conc}")
print(f"   RMSE: {mejor_rmse_conc:.4f} °C  |  R²: {mejor_r2_conc:.4f}  |  MAE: {mejor_mae_conc:.4f} °C")
print(f"\n📌 Mejor F1 MHW derivada                   : {mejor_f1_val:.4f}")
print(f"   Zona: {mejor_f1_zona}  |  Modelo: {mejor_f1_mod}")
print(f"   Precision: {mejor_precision:.4f}  |  Recall: {mejor_recall:.4f}")

if tiene_comparacion:
    print(f"\n📌 Eventos MHW observados (comparación)    : {n_ev_obs_comp}")
    print(f"📌 Eventos MHW predichos  (comparación)    : {n_ev_pred_comp}")
    print(f"📌 Días MHW observados    (comparación)    : {n_dias_obs_comp}")
    print(f"📌 Días MHW predichos     (comparación)    : {n_dias_pred_comp}")

print("\n" + "=" * 65)
print("  TABLA: MEJORES MODELOS POR ZONA")
print("=" * 65)
display(resumen_mejores.round(4))

print("\n✅ Conclusiones generadas con valores reales del notebook.")

In [ ]:
# ============================================================
# BLOQUE DASHBOARD — PEGAR AL FINAL DEL NOTEBOOK
# Genera: docs/dashboard.html + docs/index.html
# Toma TODAS las variables ya calculadas en el notebook
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import folium
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ─────────────────────────────────────────────
# A. INFORMACIÓN DEL EQUIPO
# ─────────────────────────────────────────────
TITULO = "Predicción de SST y Detección de Olas de Calor Marinas — Costa del Perú (1990–2025)"
CURSO  = "Ciencia de Datos Ambientales · Universidad de Ingeniería y Tecnología — UTEC"
AUTORES = [
    "Aguilar Quispe, Nicoll Yrene",
    "Alejo Huamani, Lizett Avelina",
    "Condezo Inocencio, Lucila Carolina",
    "Velarde Salazar, María Fernanda",
]
FUENTES = "NOAA OISST v2.1 · ICEN (ENFEN/IGP) · ERA5 (Copernicus C3S)"
METODO  = "Metodología: Hobday et al. (2016) · Modelo: Random Forest Regressor"

# ─────────────────────────────────────────────
# B. EXTRAER KPIs DEL NOTEBOOK
# ─────────────────────────────────────────────
n_eventos_hist   = len(eventos_mhw)
n_dias_mhw_hist  = int(mhw_diario['mhw'].sum())
zona_mas_dias    = resumen_anual.groupby('zona')['dias_mhw'].sum().idxmax()

top3_anios = (
    resumen_anual.groupby('anio')['dias_mhw'].sum()
    .sort_values(ascending=False).head(3).index.tolist()
)

mejor_global     = tabla_modelos.sort_values('rmse_sst').iloc[0]
mejor_zona_g     = mejor_global['zona']
mejor_modelo_g   = mejor_global['modelo']
mejor_rmse_g     = round(mejor_global['rmse_sst'], 4)
mejor_r2_g       = round(mejor_global['r2_sst'],   4)
mejor_f1_g       = round(tabla_modelos['f1_mhw_derivada'].max(), 4)

faltantes_sst    = int(df_sst_zonas['sst_promedio'].isna().sum())

n_ev_pred = len(eventos_predichos_mejor) if len(eventos_predichos_mejor) > 0 else 0

kpis = [
    ("Eventos MHW históricos",         str(n_eventos_hist)),
    ("Días MHW (1990–2025)",           str(n_dias_mhw_hist)),
    ("Zona con más días MHW",          zona_mas_dias),
    ("Años con más días MHW",          " · ".join(str(a) for a in top3_anios)),
    ("Mejor RMSE SST predicción",      f"{mejor_rmse_g} °C"),
    ("Mejor R² SST",                   str(mejor_r2_g)),
    ("Mejor F1 MHW derivada",          str(mejor_f1_g)),
    ("Eventos MHW predichos (prueba)", str(n_ev_pred)),
]

kpis_html = "".join(
    f'<div class="kpi"><div class="val">{v}</div><div class="lbl">{k}</div></div>'
    for k, v in kpis
)

# ─────────────────────────────────────────────
# C. MAPA FOLIUM
# ─────────────────────────────────────────────
mapa = folium.Map(location=[-10, -80], zoom_start=5, tiles="CartoDB positron")

# Rectángulo general
folium.Rectangle(
    bounds=[[-20, -84], [1, -76]],
    color="#1a3a5c", fill=True, fill_opacity=0.04,
    tooltip="Área de estudio — Costa del Perú"
).add_to(mapa)

# Subzonas
for info in [
    {"nombre": "Zona Norte (0°–6°S)",   "bounds": [[-6, -84],  [1,   -76]], "color": "#e74c3c"},
    {"nombre": "Zona Centro (6°–13°S)", "bounds": [[-13, -84], [-6,  -76]], "color": "#f39c12"},
    {"nombre": "Zona Sur (13°–20°S)",   "bounds": [[-20, -84], [-13, -76]], "color": "#2980b9"},
]:
    folium.Rectangle(
        bounds=info["bounds"], color=info["color"],
        fill=True, fill_opacity=0.20,
        tooltip=info["nombre"]
    ).add_to(mapa)

# Ciudades de referencia costera
for c in [
    {"n": "Tumbes",      "lat": -3.57,  "lon": -80.45},
    {"n": "Piura",       "lat": -5.20,  "lon": -80.63},
    {"n": "Lima/Callao", "lat": -12.06, "lon": -77.14},
    {"n": "Ica",         "lat": -14.07, "lon": -75.73},
    {"n": "Ilo",         "lat": -17.65, "lon": -71.34},
]:
    folium.Marker(
        location=[c["lat"], c["lon"]], popup=c["n"], tooltip=c["n"],
        icon=folium.Icon(color="darkblue", icon="info-sign")
    ).add_to(mapa)

mapa_html = mapa._repr_html_()

# ─────────────────────────────────────────────
# D. GRÁFICO 1 — Serie de tiempo SST por zona
#    (usa sst_mensual_zonas del notebook)
# ─────────────────────────────────────────────
colores_zona = {"Norte": "#e74c3c", "Centro": "#f39c12", "Sur": "#2980b9", "Peru_total": "#7f8c8d"}

fig_sst_ts = go.Figure()
for zona in ["Norte", "Centro", "Sur", "Peru_total"]:
    sub = sst_mensual_zonas[sst_mensual_zonas["zona"] == zona]
    fig_sst_ts.add_trace(go.Scatter(
        x=sub["fecha"], y=sub["sst_promedio"],
        mode="lines", name=zona,
        line=dict(color=colores_zona.get(zona, "gray"), width=1.4)
    ))
fig_sst_ts.update_layout(
    title="Serie mensual de SST promedio por zona — Costa del Perú (1990–2025)",
    xaxis_title="Fecha", yaxis_title="SST (°C)",
    template="plotly_white", height=380, legend_title="Zona"
)

# ─────────────────────────────────────────────
# E. GRÁFICO 2 — Días MHW anuales por zona
# ─────────────────────────────────────────────
fig_mhw = go.Figure()
for zona in ["Norte", "Centro", "Sur"]:
    sub = resumen_anual[resumen_anual["zona"] == zona].sort_values("anio")
    fig_mhw.add_trace(go.Bar(
        x=sub["anio"], y=sub["dias_mhw"],
        name=zona, marker_color=colores_zona[zona], opacity=0.85
    ))
fig_mhw.update_layout(
    title="Días bajo condición MHW por año y zona (1990–2025)",
    xaxis_title="Año", yaxis_title="Días MHW",
    barmode="group", template="plotly_white", height=360, legend_title="Zona"
)

# ─────────────────────────────────────────────
# F. GRÁFICO 3 — SST observada vs predicha
#    (usa predicciones_mejor_modelo)
# ─────────────────────────────────────────────
fig_pred = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=["Zona Norte", "Zona Centro", "Zona Sur"]
)
for i, zona in enumerate(["Norte", "Centro", "Sur"], start=1):
    sub = predicciones_mejor_modelo[predicciones_mejor_modelo["zona"] == zona].copy()
    fig_pred.add_trace(go.Scatter(
        x=sub["fecha"], y=sub["sst_observada"],
        name="Observada" if i == 1 else None,
        legendgroup="obs", showlegend=(i == 1),
        line=dict(color="#1a3a5c", width=1.6)
    ), row=i, col=1)
    fig_pred.add_trace(go.Scatter(
        x=sub["fecha"], y=sub["sst_predicha"],
        name="Predicha (RF)" if i == 1 else None,
        legendgroup="pred", showlegend=(i == 1),
        line=dict(color="tomato", width=1.6, dash="dash")
    ), row=i, col=1)
    fig_pred.add_trace(go.Scatter(
        x=sub["fecha"], y=sub["umbral_p90"],
        name="Umbral P90" if i == 1 else None,
        legendgroup="p90", showlegend=(i == 1),
        line=dict(color="green", width=1.2, dash="dot")
    ), row=i, col=1)
    # Sombrear días MHW observados
    mhw_obs_days = sub[sub["mhw_observada"] == True]
    fig_pred.add_trace(go.Scatter(
        x=pd.concat([mhw_obs_days["fecha"], mhw_obs_days["fecha"].iloc[::-1]]),
        y=pd.concat([mhw_obs_days["sst_observada"], mhw_obs_days["umbral_p90"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(231,76,60,0.2)",
        line=dict(color="rgba(0,0,0,0)"),
        name="MHW observada" if i == 1 else None,
        legendgroup="mhwobs", showlegend=(i == 1)
    ), row=i, col=1)

fig_pred.update_layout(
    title="SST observada vs predicha (periodo de prueba) con umbral P90 y MHW",
    height=650, template="plotly_white", legend_title="Serie"
)
fig_pred.update_yaxes(title_text="SST (°C)")

# ─────────────────────────────────────────────
# G. GRÁFICO 4 — RMSE y F1 comparación modelos
# ─────────────────────────────────────────────
fig_metricas = make_subplots(
    rows=1, cols=2,
    subplot_titles=["RMSE SST por modelo y zona (↓ mejor)", "F1 MHW derivada (↑ mejor)"]
)
for zona in ["Norte", "Centro", "Sur"]:
    sub = tabla_modelos[tabla_modelos["zona"] == zona]
    fig_metricas.add_trace(go.Bar(
        x=sub["modelo"], y=sub["rmse_sst"],
        name=zona, marker_color=colores_zona[zona],
        legendgroup=zona, showlegend=True
    ), row=1, col=1)
    fig_metricas.add_trace(go.Bar(
        x=sub["modelo"], y=sub["f1_mhw_derivada"],
        name=zona, marker_color=colores_zona[zona],
        legendgroup=zona, showlegend=False
    ), row=1, col=2)

fig_metricas.update_layout(
    barmode="group", template="plotly_white",
    height=380, legend_title="Zona"
)
fig_metricas.update_yaxes(title_text="RMSE (°C)", row=1, col=1)
fig_metricas.update_yaxes(title_text="F1", row=1, col=2)

# ─────────────────────────────────────────────
# H. GRÁFICO 5 — Importancia de variables
# ─────────────────────────────────────────────
fig_imp = px.bar(
    importancias_mejores.groupby("zona").head(10).sort_values(["zona", "importancia"]),
    x="importancia", y="feature", color="zona",
    facet_col="zona", orientation="h",
    title="Top 10 variables más importantes por zona — Random Forest Regressor",
    color_discrete_map=colores_zona,
    template="plotly_white", height=420,
    labels={"importancia": "Importancia (Gini)", "feature": "Variable"}
)
fig_imp.update_yaxes(matches=None)

# ─────────────────────────────────────────────
# I. TABLAS HTML
# ─────────────────────────────────────────────
def df_to_html(df, cols=None):
    d = df[cols].copy() if cols else df.copy()
    # Formatear fechas si las hay
    for col in d.columns:
        if pd.api.types.is_datetime64_any_dtype(d[col]):
            d[col] = d[col].dt.strftime("%Y-%m-%d")
    return d.round(4).to_html(
        index=False, border=0, classes="tbl"
    )

# Tabla: mejores modelos por zona
tabla_mejores_html = df_to_html(mejores_por_zona, cols=[
    "zona", "modelo", "mae_sst", "rmse_sst", "r2_sst",
    "precision_mhw_derivada", "recall_mhw_derivada", "f1_mhw_derivada"
])

# Tabla: top 10 eventos MHW históricos
cols_ev = ["zona", "fecha_inicio", "fecha_fin", "duracion_dias",
           "intensidad_maxima", "intensidad_acumulada", "categoria_intensidad"]
cols_ev_ok = [c for c in cols_ev if c in eventos_mhw.columns]
top10_ev_html = df_to_html(
    eventos_mhw[cols_ev_ok]
    .sort_values("intensidad_acumulada", ascending=False)
    .head(10)
)

# Tabla: eventos MHW predichos (periodo de prueba)
if len(eventos_predichos_mejor) > 0:
    cols_ep = [c for c in cols_ev if c in eventos_predichos_mejor.columns]
    tabla_pred_html = df_to_html(eventos_predichos_mejor[cols_ep])
else:
    tabla_pred_html = "<p><em>No se detectaron eventos MHW en la SST predicha durante el periodo de prueba.</em></p>"

# Tabla EDA: estadísticos SST por zona
eda_stats = df_sst_zonas.groupby("zona")["sst_promedio"].describe().round(3)
eda_html = eda_stats.to_html(classes="tbl", border=0)

# ─────────────────────────────────────────────
# J. CONSTRUIR HTML FINAL
# ─────────────────────────────────────────────
autores_html = " &nbsp;·&nbsp; ".join(AUTORES)

HTML = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <title>Dashboard MHW — Costa del Perú</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
  <style>
    :root{{
      --azul:#1a3a5c; --azul2:#2980b9; --rojo:#e74c3c;
      --naranja:#f39c12; --gris:#f4f6f9; --borde:#dde1e7;
    }}
    *{{box-sizing:border-box;margin:0;padding:0}}
    body{{font-family:'Inter',sans-serif;background:var(--gris);color:#2c3e50}}

    /* HEADER */
    header{{background:linear-gradient(135deg,var(--azul),var(--azul2));
            color:#fff;padding:32px 48px}}
    header h1{{font-size:1.5rem;font-weight:700;line-height:1.35;margin-bottom:10px}}
    header p{{font-size:.9rem;opacity:.92;margin-top:5px}}
    .team{{margin-top:14px;background:rgba(255,255,255,.15);
           border-radius:8px;padding:12px 18px;font-size:.88rem}}
    .team strong{{display:block;margin-bottom:6px}}

    /* NAV */
    nav{{background:#fff;border-bottom:2px solid var(--borde);
         padding:0 48px;position:sticky;top:0;z-index:900;
         display:flex;gap:2px;overflow-x:auto}}
    nav a{{color:var(--azul);text-decoration:none;font-size:.85rem;
           font-weight:600;padding:13px 15px;white-space:nowrap;
           border-bottom:3px solid transparent;transition:.2s}}
    nav a:hover{{border-bottom-color:var(--azul2);color:var(--azul2)}}

    /* MAIN */
    main{{max-width:1200px;margin:0 auto;padding:28px 20px}}
    section{{background:#fff;border-radius:12px;padding:26px 30px;
             margin-bottom:26px;box-shadow:0 2px 8px rgba(0,0,0,.07)}}
    section h2{{color:var(--azul);font-size:1.15rem;font-weight:700;
                border-bottom:2px solid var(--azul2);padding-bottom:9px;
                margin-bottom:16px}}
    section h3{{color:var(--azul);font-size:.98rem;margin:16px 0 8px}}
    p{{line-height:1.7;font-size:.9rem;margin-bottom:8px}}
    ul{{padding-left:20px}}
    li{{margin-bottom:6px;line-height:1.65;font-size:.9rem}}

    /* KPIs */
    .kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;margin:16px 0}}
    .kpi{{background:var(--gris);border:1px solid var(--borde);
          border-radius:10px;padding:16px 12px;text-align:center}}
    .kpi .val{{font-size:1.15rem;font-weight:700;color:var(--azul2)}}
    .kpi .lbl{{font-size:.75rem;color:#666;margin-top:5px;line-height:1.4}}

    /* TABLAS */
    .tbl{{width:100%;border-collapse:collapse;font-size:.82rem;margin-top:10px}}
    .tbl th{{background:var(--azul);color:#fff;padding:9px 11px;text-align:left;font-weight:600}}
    .tbl td{{padding:7px 11px;border-bottom:1px solid var(--borde)}}
    .tbl tr:nth-child(even){{background:var(--gris)}}
    .tbl tr:hover{{background:#e8f4fd}}
    .tabla-scroll{{overflow-x:auto}}

    /* CHIPS */
    .chips{{display:flex;gap:10px;flex-wrap:wrap;margin:10px 0}}
    .chip{{padding:4px 12px;border-radius:20px;font-size:.8rem;font-weight:600}}
    .cn{{background:#fde8e8;color:var(--rojo)}}
    .cc{{background:#fef3e2;color:#d68910}}
    .cs{{background:#eaf4fb;color:var(--azul2)}}

    /* CALLOUT */
    .box{{background:#eaf4fb;border-left:4px solid var(--azul2);
          padding:13px 17px;border-radius:0 8px 8px 0;margin:12px 0;font-size:.9rem}}

    /* FOOTER */
    footer{{background:var(--azul);color:rgba(255,255,255,.88);
            padding:22px 48px;font-size:.82rem;line-height:1.9}}

    @media(max-width:768px){{
      header,footer{{padding:22px 18px}}
      main{{padding:14px 10px}}
      nav{{padding:0 10px}}
      section{{padding:18px 14px}}
    }}
  </style>
</head>
<body>

<!-- HEADER -->
<header>
  <h1>🌊 {TITULO}</h1>
  <p>{CURSO}</p>
  <p>{METODO}</p>
  <div class="team">
    <strong>👥 Equipo:</strong>
    {autores_html}
  </div>
  <p style="margin-top:10px;font-size:.8rem;opacity:.8">📦 Fuentes: {FUENTES}</p>
</header>

<!-- NAVEGACIÓN -->
<nav>
  <a href="#resumen">Resumen</a>
  <a href="#problema">Problema</a>
  <a href="#datos">Datos</a>
  <a href="#eda">EDA</a>
  <a href="#metodologia">Metodología</a>
  <a href="#modelo">Modelo RF</a>
  <a href="#resultados">Resultados</a>
  <a href="#conclusiones">Conclusiones</a>
</nav>

<!-- MAIN -->
<main>

<!-- 0. RESUMEN -->
<section id="resumen">
  <h2>📋 Resumen ejecutivo</h2>
  <p>
    Este dashboard presenta el flujo completo del Proyecto Integrador:
    predicción de temperatura superficial del mar (SST) con <strong>Random Forest Regressor</strong>
    y detección de olas de calor marinas con la metodología de <strong>Hobday et al. (2016)</strong>,
    aplicada sobre la SST observada y sobre la SST predicha para comparar eventos.
  </p>
  <div class="kpis">{kpis_html}</div>
  <div class="box">
    <strong>Zonas de análisis:</strong>
    <span style="display:inline-flex;gap:8px;flex-wrap:wrap;margin-top:6px">
      <span class="chip cn">Norte: 0°–6°S</span>
      <span class="chip cc">Centro: 6°–13°S</span>
      <span class="chip cs">Sur: 13°–20°S</span>
    </span>
    &nbsp;|&nbsp; Periodo: 1990–2025 &nbsp;|&nbsp; Datos diarios NOAA OISST v2.1 (0.25°×0.25°)
  </div>
</section>

<!-- 1. PROBLEMA -->
<section id="problema">
  <h2>1. Descripción del problema ambiental</h2>
  <p>
    Las <strong>Olas de Calor Marinas (MHW)</strong> son periodos prolongados de temperatura
    superficial del mar anormalmente alta. En la costa peruana, en el contexto de la
    <strong>Corriente de Humboldt</strong>, estos eventos pueden alterar los ecosistemas marinos,
    reducir la productividad pesquera, modificar el afloramiento costero y asociarse a
    condiciones extremas vinculadas a El Niño.
  </p>
  <p>
    El proyecto busca <strong>predecir la SST diaria</strong> por zonas costeras y, a partir de
    esa predicción, aplicar Hobday para identificar posibles eventos MHW, generando una base
    reproducible para monitoreo y alerta temprana.
  </p>
  <h3>Indicadores ambientales clave</h3>
  <div class="tabla-scroll">
  <table class="tbl">
    <tr><th>Indicador</th><th>Unidad</th><th>Uso en el proyecto</th></tr>
    <tr><td>SST promedio zonal</td><td>°C</td><td>Variable objetivo del modelo RF</td></tr>
    <tr><td>Anomalía climatológica SST</td><td>°C</td><td>Desviación respecto al promedio histórico del día del año</td></tr>
    <tr><td>Días bajo condición MHW</td><td>días/año</td><td>Frecuencia anual de olas de calor por zona</td></tr>
    <tr><td>Intensidad acumulada del evento</td><td>°C·día</td><td>Severidad total de cada MHW detectada</td></tr>
  </table>
  </div>
  <h3>Hipótesis de trabajo</h3>
  <ul>
    <li>La SST puede ser estimada a partir de su comportamiento previo, variables estacionales e ICEN rezagado.</li>
    <li>El desempeño del modelo varía entre Norte, Centro y Sur por diferencias oceanográficas regionales.</li>
    <li>La detección MHW derivada de la SST predicha depende de la precisión del modelo, especialmente cuando la SST está cerca del umbral P90.</li>
  </ul>
</section>

<!-- 2. DATOS -->
<section id="datos">
  <h2>2. Fuentes de datos y arquitectura</h2>
  <div class="tabla-scroll">
  <table class="tbl">
    <tr><th>Fuente</th><th>Variable</th><th>Formato</th><th>Resolución</th><th>Periodo</th></tr>
    <tr><td>NOAA OISST v2.1</td><td>SST diaria</td><td>NetCDF (.nc)</td><td>0.25°×0.25° diario</td><td>1990–2025</td></tr>
    <tr><td>ENFEN / IGP</td><td>ICEN mensual</td><td>CSV</td><td>Mensual</td><td>1990–2025</td></tr>
    <tr><td>ERA5 — Copernicus</td><td>Viento u10, v10</td><td>NetCDF (.nc)</td><td>0.25°×0.25°</td><td>1990–2025</td></tr>
  </table>
  </div>
  <h3>Pipeline de procesamiento</h3>
  <div class="box">
    Carga NetCDF → Promedio espacial por zona → Serie diaria 1990–2025 →
    Climatología diaria P90 (Hobday) → Detección MHW observada →
    Features ML (lags 1–30 d, medias móviles, ICEN rezagado) →
    RF Regressor 70/30 cronológico → SST predicha →
    Hobday sobre SST predicha → Comparación observado vs predicho → Dashboard
  </div>
  <h3>Mapa interactivo del área de estudio</h3>
  <p style="font-size:.82rem;color:#666;margin-bottom:8px">
    Las franjas coloreadas representan las zonas de promedio espacial usadas en el análisis.
  </p>
  {mapa_html}
</section>

<!-- 3. EDA -->
<section id="eda">
  <h2>3. Análisis exploratorio de datos</h2>
  <div class="box">
    Valores faltantes detectados en SST: <strong>{faltantes_sst}</strong>
    (no se eliminaron outliers automáticamente — pueden representar eventos extremos reales)
  </div>
  <h3>Estadísticos de SST por zona</h3>
  <div class="tabla-scroll">{eda_html}</div>
  <h3>Serie mensual de SST por zona (1990–2025)</h3>
  {fig_sst_ts.to_html(full_html=False, include_plotlyjs=False)}
  <h3>Días MHW anuales por zona</h3>
  {fig_mhw.to_html(full_html=False, include_plotlyjs=False)}
</section>

<!-- 4. METODOLOGÍA -->
<section id="metodologia">
  <h2>4. Metodología de detección MHW — Hobday et al. (2016)</h2>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px">
    <div class="box">
      <strong>Criterios aplicados:</strong>
      <ul style="margin-top:8px">
        <li>Climatología diaria sobre baseline 1990–2025</li>
        <li>Umbral P90 diario suavizado (ventana ±5 días, suavizado 31 días)</li>
        <li>Día candidato MHW cuando SST &gt; P90</li>
        <li>Evento MHW: mínimo 5 días consecutivos</li>
        <li>Unión de eventos separados por brechas ≤ 2 días</li>
      </ul>
    </div>
    <div class="box">
      <strong>Separación metodológica clave:</strong>
      <p style="margin-top:8px">
        El modelo RF <em>predice SST</em>, no MHW directamente.
        Hobday se aplica <em>después</em> sobre la SST predicha para comparar
        la detección con los eventos observados.
      </p>
    </div>
  </div>
</section>

<!-- 5. MODELO RF -->
<section id="modelo">
  <h2>5. Modelo Random Forest Regressor — Predicción de SST</h2>
  <p>
    Se entrenaron <strong>3 configuraciones de RF Regressor</strong> por zona con división cronológica
    70/30. Las variables predictoras incluyen rezagos de SST (1, 3, 7, 14, 30 días),
    medias móviles previas, anomalías, exceso térmico, estacionalidad (seno/coseno del día del año)
    e ICEN rezagado un mes cuando está disponible.
  </p>
  <h3>Comparación de RMSE y F1 por modelo y zona</h3>
  {fig_metricas.to_html(full_html=False, include_plotlyjs=False)}
  <h3>Importancia de variables (top 10 por zona)</h3>
  {fig_imp.to_html(full_html=False, include_plotlyjs=False)}
  <h3>Tabla: mejores modelos por zona</h3>
  <div class="tabla-scroll">{tabla_mejores_html}</div>
</section>

<!-- 6. RESULTADOS -->
<section id="resultados">
  <h2>6. Resultados — SST predicha y eventos MHW</h2>
  <p>
    Comparación de SST observada vs predicha en el periodo de prueba (último 30%).
    Las franjas rojas indican <strong>MHW observadas</strong> (SST observada sobre umbral P90).
  </p>
  {fig_pred.to_html(full_html=False, include_plotlyjs=False)}

  <h3>Top 10 eventos MHW históricos — por intensidad acumulada</h3>
  <div class="tabla-scroll">{top10_ev_html}</div>

  <h3>Eventos MHW detectados desde SST predicha (periodo de prueba)</h3>
  <div class="tabla-scroll">{tabla_pred_html}</div>
</section>

<!-- 7. CONCLUSIONES -->
<section id="conclusiones">
  <h2>7. Conclusiones, recomendaciones y limitaciones</h2>
  <h3>Síntesis de hallazgos</h3>
  <ul>
    <li>Se detectaron <strong>{n_eventos_hist} eventos MHW</strong> históricos en la costa peruana (1990–2025),
        acumulando <strong>{n_dias_mhw_hist} días</strong> bajo condición MHW.
        Los años con mayor actividad fueron <strong>{", ".join(str(a) for a in top3_anios)}</strong>.</li>
    <li>La zona <strong>{zona_mas_dias}</strong> registró la mayor frecuencia de días MHW acumulados.</li>
    <li>El mejor modelo global fue <strong>{mejor_modelo_g}</strong> en zona <strong>{mejor_zona_g}</strong>,
        con RMSE de <strong>{mejor_rmse_g} °C</strong> y R² de <strong>{mejor_r2_g}</strong>.</li>
    <li>El mejor F1 de detección MHW derivada de SST predicha fue <strong>{mejor_f1_g}</strong>,
        mostrando que errores pequeños cerca del umbral P90 pueden cambiar la clasificación de un día.</li>
  </ul>
  <h3>Recomendaciones técnicas</h3>
  <ul>
    <li>Priorizar horizontes cortos de predicción (1–7 días) para alerta temprana operativa.</li>
    <li>Actualizar el modelo periódicamente con nuevos datos observados para reducir acumulación de error.</li>
    <li>Complementar con variables oceanográficas de mayor resolución cuando estén disponibles.</li>
  </ul>
  <h3>Limitaciones</h3>
  <ul>
    <li>La predicción recursiva puede acumular errores en horizontes largos.</li>
    <li>El promedio espacial por zona puede ocultar procesos locales (afloramiento costero localizado).</li>
    <li>El ICEN es mensual y actúa como contexto climático, no como señal diaria.</li>
  </ul>
</section>

</main>

<!-- FOOTER -->
<footer>
  <strong>{TITULO}</strong><br>
  {CURSO}<br>
  👥 Equipo: {autores_html}<br>
  📦 Fuentes: {FUENTES} | Descarga de datos: enero 2025<br>
  🔬 {METODO}
</footer>

</body>
</html>"""

# ─────────────────────────────────────────────
# K. GUARDAR docs/dashboard.html e index.html
# ─────────────────────────────────────────────
DOCS = Path("docs")
DOCS.mkdir(exist_ok=True)

(DOCS / "dashboard.html").write_text(HTML, encoding="utf-8")
(DOCS / "index.html").write_text(HTML, encoding="utf-8")

print("=" * 60)
print("✅  Dashboard generado correctamente.")
print(f"    docs/dashboard.html")
print(f"    docs/index.html")
print()
print("📌  Pasos para publicar en GitHub Pages:")
print("    1. Sube la carpeta docs/ a tu repositorio GitHub")
print("    2. Settings → Pages → Source: main → /docs → Save")
print("    3. URL: https://TU-USUARIO.github.io/NOMBRE-REPO/")
print("=" * 60)

## 17. Exportacion de resultados

Se guardan las tablas principales para usarlas luego en Canva, en el informe o en el dashboard.

In [ ]:
archivos_exportar = {
    'mhw_diario_zonas_observado.csv': mhw_diario,
    'eventos_mhw_zonas_observado.csv': eventos_mhw,
    'resumen_anual_zonas_observado.csv': resumen_anual,
    'dataset_ml_prediccion_sst.csv': df_ml,
    'comparacion_rf_regressor_sst_zonas.csv': tabla_modelos,
    'mejores_modelos_sst_por_zona.csv': mejores_por_zona,
    'predicciones_test_sst_mejor_modelo.csv': predicciones_mejor_modelo,
    'diario_mhw_predicho_mejor_modelo.csv': diario_predicho_mejor,
    'eventos_mhw_predichos_mejor_modelo.csv': eventos_predichos_mejor,
    'importancias_mejores_modelos_sst.csv': importancias_mejores,
}

for nombre, df in archivos_exportar.items():
    ruta = OUT_DIR / nombre
    df.to_csv(ruta, index=False)
    print('Guardado:', ruta)